# Piece of my Heart

## Overview

This module implements the piece system for the hex-based strategy game, including:
- **Piece simulation** - Calculate future movement paths based on instruction rules
- **Visual overlays** - Render piece positions and planned movements on the map
- **Interactive controls** - Facing selector widget for piece orientation
- **Route handlers** - Web endpoints for piece detail views and maps

## Architecture

### Three-Layer Design

1. **Kingdom Directive** — Strategic intent (FEED, EXPLORE, EXPAND, DEFEND, ATTACK)
2. **Piece Orders** — Directive mapped onto each piece based on type and stats
3. **Moves** — Resolved outcomes after all kingdoms' orders collide


I am going to work through some thoughts

## Modified Gosper
so the rules on Piece is going to be reworked as a list of ints with the following
0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
4 -> settle or siege # reserved for figuring out how to conquer settlements
5 -> forward # move on place forward (depending on movement cost). If you can't move forward (ie you hit water) rotate left. if it is there is another piece in the way pause   
6 -> rotate left
7 -> rotate right


## Movement costs

we have a movement cost formula
```python
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost
```
Which we will need to figure out how to use for a turn. movement_range will tell how many moves we can do. Lets make turns free (including turns from hitting the ocean). the first 4 counts as a single movement cost.

## food and diet
we are going to add three fields 
1. which is the amount of food a piece has
2. its diet. how much does it eat per turn If it rounds out of food it will start to lose health. 
3. food capacity which is how much total food it can carry.

## deprecate
we don't need memory since that will be in rules.

## settle
We are going to keep these around even though I haven't figure them out
harvest_goal: Resources = None
settle_progress: int = 0
settle_threshold: int = 3
sight = 3

## The things we carry
Lets start thinking about the rules editor for a piece
1. we are going to want a list of rules that are assigned and be able to add and delete rows
2. we are going to want an edit point where we can insert new items
3. we are going to want a HexDragMap that we can add items. The path we make needs to be converted into instructions on how we get there.
4. we need an apply button and the number of rounds. This will be our main map showing where the piece is going to move each turn for the number of turns. I envision this as a series of arrows and circles.
a. arrows are for movement. straight line if single cost squiggle line if multiple cost. the color of the arrow is how aggressive it is.
b. circles are for the pause actions (0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
) I think red should be for defend.

eventually we are going to want a drop down of a list of common rule sets (patrol around a circle. go out and harvest and comeback, attack by first flanking left). 
we will certainly want to have a hex destination that returns a set of rules on how to get there (hello djisktra)

I need to be clearer on some definitons
1. `round` is for all player and all pieces. Each player gives the set of orders it wants its pieces to do. We need to build out how to carry them out, but I am tempted just to start with the kindergaren model of everybody does what it wants
2. `turn` is equivalent to a round, but just on a single piece. `movement_cost` is how many things it can do in a turn. Some (like rotation) are free. Others like mmoving up a hill might cost a couple of movements.

It can be the case that rule ends and there are spare items. So if the last item in the list is one of the non movement items. ie (like pause) then we just pause the remaining amount of time. if it is a movement (or rotate) then we circle back to the begining.

## Pulling Ranks
so things are prioritzed in chess order ie kings move first and eat first. then queens. This way if a king will move to a spot before a pawn would

I am going to have the flow of the hex be the amount you can harvest. Settlments are going to have build queues. I need to figure out the mechanics of adding settlements whether there are pre existing locations or it behavior on a item

**7. Food economy**

Ill need a `hex_food_yield(terrain, idx) -> float` keyed off climate and flow.



**3. Forward-blocked spin trap**

Instruction 4 says "rotate left if water, pause if piece." But imagine a piece in a cove with water on 5 sides — it rotates 5 times but each time if the chance to move becomes available



Here's where we've landed:

**Budget model** — `move_strength` (rename to `speed`) is the per-turn movement budget, not per-instruction.

| Instruction | Cost | Notes |
|---|---|---|
| `FORWARD` (5) | 1 flat/downhill, 2 uphill | Can't move if budget < cost → turn ends |
| `ROT_L` (6) / `ROT_R` (7) | free | Including auto-rotations from hitting water |
| `PAUSE` (0) | 1 | Explicit wait |
| `DEFEND` (1) | 1 | Boosts combat response |
| `HARVEST` (2) | 1 | Gathers `hex_food_yield` into `food` (capped at `food_capacity`) |
| `GIVE` (3) | 1 | Transfer food to adjacent piece or settlement |
| `SETTLE` (4) | 1 | Reserved — siege/founding |

**FORWARD special cases:**
- **Water/edge** → auto-rotate left (free), up to 5×. Full circle → stuck, forced PAUSE (costs 1)
- **Occupied hex** → forced PAUSE (costs 1), cursor still advances

**End-of-rules:**
- Last instruction is non-movement (`PAUSE`/`DEFEND`/`HARVEST`/`GIVE`/`SETTLE`) → idle out remaining budget
- Last instruction is movement/rotation → wrap cursor to 0, keep going

**Turn resolution order:**
- KING → QUEEN → BISHOP → KNIGHT → ROOK → PAWN (chess rank priority)

**Food economy:**
- `food` — current supply
- `diet` — consumed per turn (deducted at turn end)
- `food_capacity` — max carried
- `food < 0` → lose health

**Example: Pawn (speed=3), rules `F F F F F F`, flat terrain:**
```
Turn 1: F(1) F(1) F(1) → budget 0
Turn 2: F(1) F(1) F(1) → budget 0
```

**Example: Pawn (speed=3), rules `F F F F F F`, uphill at step 2:**
```
Turn 1: F(1) F_uphill(2) → budget 0 (only 2 hexes moved)
Turn 2: F(1) F(1) F(1)  → budget 0
```

**Example: Pawn (speed=3), rules `F F HARVEST`:**
```
Turn 1: F(1) F(1) HARVEST(1) → end of rules, last=HARVEST → idle
Turn 2: same pattern (wraps because patrol=True)
```



In [ ]:
#| default_exp game/piece

In [ ]:
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np
from dataclasses import dataclass

In [ ]:
from monsterui.all import *

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion



In [ ]:
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction

## Piece View — Project Plan
That's a really clean three-layer design:

1. **Kingdom Directive** — strategic intent ("we're starving", "explore", "expand")
2. **Piece Orders** — the directive gets mapped onto each piece based on its type and base stats
3. **Moves** — resolved outcomes after all kingdoms' orders collide

So the directive acts like the queen bee's pheromone signal — it shifts *what* each piece does, but *how well* they do it still depends on their base stats. A "gather food" directive would assign harvest orders to pawns (who are great at it) but might also reassign knights to patrol the borders instead of scouting.

For the directive enum, something like:

- **FEED** — prioritize harvest
- **EXPLORE** — prioritize scouting / vision
- **EXPAND** — claim new territory
- **DEFEND** — protect capital and settlements
- **ATTACK** — target enemy pieces/settlements

These look right. So we would to combine them to make a patrol concept. we could have a gosper curve that a piece would through as it moves. if we see something and it wants to attack then it could

That's a brilliant use of the Gosper curve — it's space-filling, so a piece on patrol would efficiently cover an area without a lot of backtracking. The concept would be something like:

    Generate patrol path — a Gosper curve anchored to a settlement or border, sized to fit the piece's territory
    Each turn — the piece advances along the curve by move_strength hexes
    At each hex — check sight range for triggers (enemy pieces, unclaimed resources, etc.)
    Interrupt — if something is spotted, the piece breaks off patrol and generates a new order (attack, harvest, scout closer)

So a patrol is really a streaming order — it keeps going until something more interesting happens. The directive level would control where the patrol is anchored and what triggers a break:

    DEFEND → patrol around capital, interrupt on enemy sighting → attack
    EXPLORE → patrol outward from borders, interrupt on discovery → report back
    FEED → patrol farmable land, interrupt on good hex → harvest


Nice. So let's sketch the Order shape. Something like:
Copied!

class OrderType(Enum):
    PATROL = "patrol"
    ATTACK = "attack"
    MOVE   = "move"
    HARVEST = "harvest"
    SCOUT  = "scout"




In [ ]:
app, rt, hexGameServer  = appRoutes()

## Helpers

In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

In [ ]:
!cat ../../HexMagic/game/data.py

In [ ]:
!cat ../../docs/ll*

### 1. Data Structures

In [ ]:
@dataclass
class PieceStep:
    """One resolved step in a piece's movement plan."""
    hex_idx: int
    facing: int
    instruction: Instruction
    prev_idx: int
    step: int               # global step counter (across all turns)
    turn: int               # which turn (0-indexed)
    cost: float             # movement points consumed by this step
    budget_remaining: float # budget left after this step
    blocked: bool = False
    auto_rotations: int = 0 # free rotations from hitting water/edge

# Cost categories
_COSTS_ONE = {Instruction.PAUSE, Instruction.DEFEND, Instruction.HARVEST,
              Instruction.GIVE, Instruction.SETTLE}
_FREE = {Instruction.ROT_L, Instruction.ROT_R}

def _move_cost(elevations, from_idx, to_idx) -> float:
    """Simple movement cost: 1 flat/downhill, 2 uphill."""
    diff = elevations[to_idx] - elevations[from_idx]
    return 2.0 if diff > 0 else 1.0


In [ ]:

@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_turns: int = 10,
             countries: np.ndarray = None,
             occupied: set[int] = None) -> list['PieceStep']:
    """Simulate piece over multiple turns using move_strength as per-turn budget.

    Budget rules:
      - FORWARD: costs _move_cost (1 flat, 2 uphill). If can't afford → end turn.
      - ROT_L / ROT_R: free
      - PAUSE / DEFEND / HARVEST / GIVE / SETTLE: cost 1
      - FORWARD into water/edge: auto-rotate left (free, up to 5×). Full circle → stuck, cost 1.
      - FORWARD into occupied hex: pause instead, cost 1.

    End-of-rules:
      - Last instruction was non-movement → pause out remaining budget.
      - Last instruction was movement/rotate → wrap cursor to 0.
    """
    if not self.rules or self.location is None:
        return []

    occupied = set(occupied) if occupied else set()
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor
    step_n = 0

    for turn in range(num_turns):
        budget = float(self.move_strength)

        while budget > 0:
            # --- End of rules ---
            if cur >= len(self.rules):
                last = Instruction(self.rules[-1])
                if last in _COSTS_ONE:
                    # Idle out the turn
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE,
                                           pos, step_n, turn, budget, 0.0))
                    step_n += 1
                    break
                cur = 0  # wrap around

            instr = Instruction(self.rules[cur])
            prev = pos

            # --- FREE: rotations ---
            if instr in _FREE:
                fac = ((fac - 1) if instr == Instruction.ROT_L
                       else (fac + 1)) % 6
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 0.0, budget))
                step_n += 1; cur += 1
                continue

            # --- COST 1: non-movement actions ---
            if instr in _COSTS_ONE:
                budget -= 1.0
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 1.0, budget))
                step_n += 1; cur += 1
                continue

            # --- FORWARD ---
            auto_rots = 0
            moved = False
            saved_fac = fac

            while auto_rots <= 5:
                d = HexPosition.directions()[fac % 6]
                nbr = grid.hexposition_to_index(d, pos)

                passable = (0 <= nbr < len(elevations)
                            and nbr not in grid.invalidRegion
                            and elevations[nbr] >= 1)
                if passable and countries is not None:
                    passable = countries[nbr] >= 0

                if not passable:
                    fac = (fac - 1) % 6   # free auto-rotate left
                    auto_rots += 1
                    continue

                # Occupied → forced pause
                if nbr in occupied:
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                           step_n, turn, 1.0, budget - 1.0,
                                           blocked=True,
                                           auto_rotations=auto_rots))
                    budget -= 1.0
                    step_n += 1; cur += 1
                    moved = True
                    break

                # Can we afford it?
                cost = _move_cost(elevations, pos, nbr)
                if cost > budget:
                    budget = 0       # turn over, can't afford
                    moved = True
                    break

                # Move
                occupied.discard(pos)
                pos = nbr
                occupied.add(pos)
                budget -= cost
                steps.append(PieceStep(pos, fac, Instruction.FORWARD, prev,
                                       step_n, turn, cost, budget,
                                       auto_rotations=auto_rots))
                step_n += 1; cur += 1
                moved = True
                break

            if not moved:
                # Spun full circle — stuck
                fac = saved_fac      # 6 left rotations = back to original
                steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                       step_n, turn, 1.0, budget - 1.0,
                                       blocked=True, auto_rotations=6))
                budget -= 1.0
                step_n += 1; cur += 1

    return steps


def steps_by_turn(steps: list[PieceStep]) -> dict[int, list[PieceStep]]:
    """Group steps by turn number."""
    turns = {}
    for s in steps:
        turns.setdefault(s.turn, []).append(s)
    return turns


@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_turns: int = 10, padding: int = 3,
                countries: np.ndarray = None,
                occupied: set[int] = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_turns,
                          countries=countries, occupied=occupied)
    visited = {self.location}
    for s in steps:
        visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations):
                expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)


### 2. Glyphs

In [ ]:
#| export
@dataclass
class DiagramGlyphs:
    """Shared SVG glyph renderer for piece-plan overlays."""
    color:        str   = "#333"
    stroke_width: float = 1.5
    opacity:      float = 0.85
    size:         float = 10.0


@patch
def circle(self: DiagramGlyphs, cx, cy, r, fill="none", stroke=None,
           stroke_width=None, opacity=None, **kw):
    stroke = stroke or self.color
    stroke_width = stroke_width or self.stroke_width
    opacity = opacity if opacity is not None else self.opacity
    return (f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{r:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}"/>\n')


@patch
def ring(self: DiagramGlyphs, cx, cy, r, stroke=None, stroke_width=2, opacity=0.9):
    return self.circle(cx, cy, r, fill="none",
                       stroke=stroke or self.color,
                       stroke_width=stroke_width, opacity=opacity)


@patch
def arc(self: DiagramGlyphs, cx, cy, r, start_deg, end_deg,
        stroke=None, stroke_width=2, opacity=0.8):
    import math
    stroke = stroke or self.color
    s, e = math.radians(start_deg), math.radians(end_deg)
    x1, y1 = cx + r * math.cos(s), cy + r * math.sin(s)
    x2, y2 = cx + r * math.cos(e), cy + r * math.sin(e)
    large = 1 if (end_deg - start_deg) % 360 > 180 else 0
    return (f'<path d="M {x1:.1f},{y1:.1f} A {r:.1f},{r:.1f} 0 {large} 1 '
            f'{x2:.1f},{y2:.1f}" fill="none" stroke="{stroke}" '
            f'stroke-width="{stroke_width}" opacity="{opacity}" '
            f'stroke-linecap="round"/>\n')


@patch
def shield(self: DiagramGlyphs, cx, cy, size=None, fill=None,
           stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    w, h = size, size * 1.2
    return (f'<path d="M {cx:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy+h*0.1:.1f} '
            f'Q {cx:.1f},{cy+h*0.5:.1f} {cx-w*0.5:.1f},{cy+h*0.1:.1f} '
            f'L {cx-w*0.5:.1f},{cy-h*0.5:.1f} Z" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
            f'opacity="{opacity}"/>\n')


@patch
def sheaf(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size
    svg = ""
    for dx in [-s*0.3, 0, s*0.3]:
        svg += (f'<line x1="{cx+dx:.1f}" y1="{cy-s*0.5:.1f}" '
                f'x2="{cx+dx*0.3:.1f}" y2="{cy+s*0.5:.1f}" '
                f'stroke="{fill}" stroke-width="{max(1.5, s*0.15):.1f}" '
                f'stroke-linecap="round" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.35:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.35:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="{max(1, s*0.1):.1f}" '
            f'opacity="{opacity}"/>\n')
    return svg


@patch
def gift(self: DiagramGlyphs, cx, cy, size=None, fill=None,
         stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size * 0.8
    svg = (f'<rect x="{cx-s*0.5:.1f}" y="{cy-s*0.4:.1f}" '
           f'width="{s:.1f}" height="{s*0.8:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'rx="{s*0.1:.1f}" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx:.1f}" y1="{cy-s*0.4:.1f}" '
            f'x2="{cx:.1f}" y2="{cy+s*0.4:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.5:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.5:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg


@patch
def house(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size
    svg = (f'<rect x="{cx-s*0.4:.1f}" y="{cy:.1f}" '
           f'width="{s*0.8:.1f}" height="{s*0.5:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'opacity="{opacity}"/>\n')
    svg += (f'<polygon points="'
            f'{cx-s*0.5:.1f},{cy:.1f} '
            f'{cx:.1f},{cy-s*0.45:.1f} '
            f'{cx+s*0.5:.1f},{cy:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
            f'opacity="{opacity}"/>\n')
    return svg


@patch
def pause(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=0.7):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    w, h = size * 0.2, size * 0.6
    svg = ""
    for dx in [-size*0.18, size*0.18]:
        svg += (f'<rect x="{cx+dx-w/2:.1f}" y="{cy-h/2:.1f}" '
                f'width="{w:.1f}" height="{h:.1f}" '
                f'fill="{fill}" stroke="{stroke}" stroke-width="1" '
                f'rx="{w*0.2:.1f}" opacity="{opacity}"/>\n')
    return svg


@patch
def rot_arrow(self: DiagramGlyphs, cx, cy, r=None, clockwise=True,
              stroke=None, stroke_width=2, opacity=None):
    import math
    r = r or self.size * 0.5
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    if clockwise:
        arc_svg = self.arc(cx, cy, r, -120, 150, stroke, stroke_width, opacity)
        tip_x = cx + r * math.cos(math.radians(150))
        tip_y = cy + r * math.sin(math.radians(150))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x-r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x-r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    else:
        arc_svg = self.arc(cx, cy, r, 30, 300, stroke, stroke_width, opacity)
        tip_x = cx + r * math.cos(math.radians(30))
        tip_y = cy + r * math.sin(math.radians(30))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x+r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x+r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    return arc_svg + ah


In [ ]:
@patch
def blocked_x(self: DiagramGlyphs, cx, cy, r=None) -> str:
    """Red circle with white X cross — blocked/impassable move marker."""
    r = r or self.size * 0.25
    svg = self.circle(cx, cy, r, fill="red", stroke="darkred",
                      stroke_width=1, opacity=0.55)
    d = r * 0.55
    for x1, y1, x2, y2 in [(cx-d, cy-d, cx+d, cy+d),
                             (cx+d, cy-d, cx-d, cy+d)]:
        svg += (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
                f'x2="{x2:.1f}" y2="{y2:.1f}" '
                f'stroke="white" stroke-width="{max(1, r*0.35):.1f}" '
                f'stroke-linecap="round"/>\n')
    return svg


@patch
def start_marker(self: DiagramGlyphs, cx, cy, r=None,
                 stroke=None, stroke_width=None) -> str:
    """Hollow ring drawn around the piece's start position."""
    r = r or self.size * 0.8
    return self.ring(cx, cy, r,
                     stroke=stroke or self.color,
                     stroke_width=stroke_width or max(1.5, self.size * 0.07))


@patch
def step_label(self: DiagramGlyphs, cx, cy, n: int,
               offset=None, font_size=None, color=None) -> str:
    """Small step-number badge offset up-right from a hex centre.
    
    Args:
        n: step number (0-indexed internally; displayed as n+1)
        offset: pixel offset from centre (defaults to size * 0.35)
        font_size: override font size (defaults to max(12, size * 0.3))
        color: text color (defaults to self.comp if present, else self.color)
    """
    offset    = offset    or self.size * 0.35
    font_size = font_size or max(12, self.size * 0.3)
    color     = color     or getattr(self, 'comp', self.color)
    return (f'<text x="{cx+offset:.1f}" y="{cy-offset:.1f}" '
            f'text-anchor="start" font-size="{font_size:.0f}" '
            f'fill="{color}" font-family="sans-serif" '
            f'font-weight="bold">{n+1}</text>\n')


@patch
def action_glyph(self: DiagramGlyphs, instr: Instruction,
                 cx, cy, fill=None, stroke=None) -> str:
    """Dispatch a non-movement instruction to its geometry glyph.
    
    Returns empty string for movement instructions (FORWARD) — those
    are handled separately as arrows.
    """
    fill   = fill   or self.color
    stroke = stroke or getattr(self, 'comp', self.color)
    size   = self.size

    dispatch = {
        Instruction.PAUSE:   lambda: self.pause(cx, cy, size, fill, stroke),
        Instruction.DEFEND:  lambda: self.shield(cx, cy, size, fill, stroke),
        Instruction.HARVEST: lambda: self.sheaf(cx, cy, size, fill, stroke),
        Instruction.GIVE:    lambda: self.gift(cx, cy, size, fill, stroke),
        Instruction.SETTLE:  lambda: self.house(cx, cy, size, fill, stroke),
        Instruction.ROT_L:   lambda: self.rot_arrow(cx, cy, clockwise=False,
                                                     stroke=stroke),
        Instruction.ROT_R:   lambda: self.rot_arrow(cx, cy, clockwise=True,
                                                     stroke=stroke),
    }
    fn = dispatch.get(instr)
    return fn() if fn else ""


### 3. Instruction extensions

In [ ]:
#| export
INSTR_LABELS = {
    Instruction.PAUSE:   ("Pause",      "⏸"),
    Instruction.DEFEND:  ("Defend",     "🛡"),
    Instruction.HARVEST: ("Harvest",    "🍆"),
    Instruction.GIVE:    ("Give",       "🎁"),
    Instruction.SETTLE:  ("Settle",     "🏰"),
    Instruction.FORWARD: ("Forward",    "→"),
    Instruction.ROT_L:   ("Turn Left",  "↺"),
    Instruction.ROT_R:   ("Turn Right", "↻"),
}


def _instr_glyph(instr: Instruction, size=26, color="#444"):
    """Inline SVG glyph for one instruction."""
    g = DiagramGlyphs(color=color, size=size * 0.55)
    cx, cy = size / 2, size / 2
    if instr == Instruction.FORWARD:
        body = (f'<line x1="{cx-size*0.28:.1f}" y1="{cy:.1f}" '
                f'x2="{cx+size*0.2:.1f}" y2="{cy:.1f}" '
                f'stroke="{color}" stroke-width="2.5" stroke-linecap="round"/>'
                f'<polygon points="{cx+size*0.28:.1f},{cy:.1f} '
                f'{cx+size*0.12:.1f},{cy-size*0.13:.1f} '
                f'{cx+size*0.12:.1f},{cy+size*0.13:.1f}" fill="{color}"/>')
    else:
        body = g.action_glyph(instr, cx, cy) or ""
    return NotStr(f'<svg width="{size}" height="{size}" viewBox="0 0 {size} {size}" '
                  f'style="vertical-align:middle">{body}</svg>')


In [ ]:
class InstructionList:
    """Displayable, serialisable list of piece instructions with glyphs and step numbers."""

    def __init__(self, rules: list[int], cursor: int = 0, patrol: bool = True):
        self.rules  = rules
        self.cursor = cursor
        self.patrol = patrol

    # ── Rendering ─────────────────────────────────────────────

    def __ft__(self):
        """Full MonsterUI table with SVG glyphs and cursor marker."""
        if not self.rules:
            return P("No instructions", cls=TextPresets.muted_sm)

        header = ["#", "", "Action"]
        rows   = []
        for i, r in enumerate(self.rules):
            instr = Instruction(r)
            label, _ = INSTR_LABELS.get(instr, (instr.name, "?"))
            step = Strong("▶ ", cls="text-primary") if i == self.cursor else str(i + 1)
            rows.append([step, _instr_glyph(instr), label])

        tbl  = TableFromLists(header, rows, cls=(TableT.sm, TableT.striped, TableT.hover))
        mode = "🔁 Patrol (loops)" if self.patrol else "⏹ One-shot"
        return Card(tbl, footer=Small(mode, cls=TextPresets.muted_sm))

    def compact(self, piece_id: str = "") -> FT:
        """Compact inline badge strip — lightweight alternative to the full table."""
        target_id = f"rules-{piece_id[:8]}" if piece_id else "rules-preview"

        if not self.rules:
            return P("No rules yet — drag a path above",
                     cls="text-sm opacity-50", id=target_id)

        _COLORS = {
            0: "#94a3b8", 1: "#ef4444", 2: "#22c55e",
            3: "#a855f7", 4: "#f59e0b", 5: "#3b82f6",
            6: "#64748b", 7: "#64748b",
        }
        badges = []
        for r in self.rules:
            instr  = Instruction(r)
            label, _ = INSTR_LABELS.get(instr, (instr.name, "?"))
            color  = _COLORS.get(r, "#888")
            badges.append(
                Span(label,
                     style=(f"background:{color}22; border:1px solid {color}; "
                            "border-radius:4px; padding:2px 6px; font-size:11px; "
                            f"color:{color}; margin:2px; display:inline-block;"))
            )

        return Div(
            P(f"{len(self.rules)} instructions", cls="text-xs opacity-50 mb-1"),
            Div(*badges, style="line-height:2"),
            id=target_id,
        )

    # ── Factories ─────────────────────────────────────────────

    @classmethod
    def from_path(cls, hex_indices: list[int], grid: HexGrid,
                  start_facing: int = 0, patrol: bool = True) -> "InstructionList":
        """Build an InstructionList directly from a dragged hex path."""
        return cls(
            rules=cls.path_to_rules(hex_indices, grid, start_facing),
            patrol=patrol,
        )

    @staticmethod
    def path_to_rules(hex_indices: list[int], grid: HexGrid,
                      start_facing: int = 0) -> list[int]:
        """Convert a list of hex grid indices into Instruction ints.

        Deduplicates consecutive repeated hexes (common in drag traces),
        then emits the minimum ROT_L / ROT_R turns needed before each FORWARD.
        Non-adjacent jumps are silently skipped.
        """
        if len(hex_indices) < 2:
            return []

        dirs = HexPosition.directions()   # [SW, W, NW, NE, E, SE] → indices 0–5

        # Remove consecutive duplicates from drag noise
        clean = [hex_indices[0]]
        for idx in hex_indices[1:]:
            if idx != clean[-1]:
                clean.append(idx)

        rules  = []
        facing = start_facing

        for i in range(len(clean) - 1):
            from_pos = grid.index_to_hexposition(clean[i],     clean[0])
            to_pos   = grid.index_to_hexposition(clean[i + 1], clean[0])
            delta    = to_pos - from_pos

            try:
                target_dir = next(j for j, d in enumerate(dirs) if d == delta)
            except StopIteration:
                continue   # non-adjacent hex — skip

            diff = (target_dir - facing) % 6
            if diff == 1:
                rules.append(Instruction.ROT_R.value)
            elif diff == 5:
                rules.append(Instruction.ROT_L.value)
            elif 2 <= diff <= 3:
                rules.extend([Instruction.ROT_R.value] * diff)
            elif diff == 4:
                rules.extend([Instruction.ROT_L.value] * 2)
            # diff == 0: already facing correctly

            rules.append(Instruction.FORWARD.value)
            facing = target_dir

        return rules


In [ ]:
@patch
def delete_at(self: InstructionList, idx: int):
    """Remove instruction at idx, clamping cursor."""
    if 0 <= idx < len(self.rules):
        self.rules.pop(idx)
        self.cursor = min(self.cursor, len(self.rules))

@patch
def insert_rules(self: InstructionList, new_rules: list[int]):
    """Insert rules at cursor position, advance cursor past them."""
    for i, r in enumerate(new_rules):
        self.rules.insert(self.cursor + i, r)
    self.cursor += len(new_rules)

@patch
def set_cursor(self: InstructionList, idx: int):
    self.cursor = max(0, min(idx, len(self.rules)))

@patch
def toggle_patrol(self: InstructionList):
    self.patrol = not self.patrol


In [ ]:
rules = [Instruction.FORWARD.value, Instruction.FORWARD.value,
         Instruction.ROT_L.value, Instruction.HARVEST.value, Instruction.SETTLE.value,
         Instruction.FORWARD.value, Instruction.ROT_R.value]

show(InstructionList(rules, cursor=2, patrol=True))


arrowG = """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 500 500"><g fill="#000"><g><path d="M458.38 204.54c-11.8-47.61-38.27-90.55-74.56-120.9-41.46-34.57-96.19-42.5-134.8-43.05-48.75-.98-93.83 19.68-132.19 41.5 10.22-11.39 20.78-22.93 29.95-35.36 2.96-3.91.39-10.74-4.39-10.25-4.84.5-8.13 8.95-11 11.23a879.86 879.86 0 0 1-26.68 31.05c-14.68 15.47-10.31 25.28 11.71 24.94 21.9.45 44.17.37 65.74.46 4.18 0 5.86-6.11 3.12-8.84-2.17-2.16-6.02-1.44-8.5-1.57-18.73-.83-37.72.02-56.43-1.12 16.5-9.21 35.76-19.33 56.38-27.4 30.92-12.31 65.47-16.43 99.96-11.91 31.31 3.29 75.69 14.26 109.32 47.85 31.96 31.39 54.2 71.65 62.62 113.36 6.25 38.78 2.22 81.74-11.65 124.26-12.52 33.82-20.08 32.82-20.07 38.09 0 3.72 5.53 6.99 8.56 3.3a88.7 88.7 0 0 0 8.74-11.25c17.66-28.96 23.69-63.78 26.95-90.95 2.98-28.38 2.07-52.4-2.77-73.44ZM401.17 395.69c-8.77-5.89-20.28-3.78-29.78-5.05-8.61-.9-31.42-1.04-49.14-3.02-2.32-1.28-4.83-.7-6.14 1.11-1.47 2.21-1.79 4.18-.94 5.88 2.84 5.7 17.26 4.45 23.82 5.68 14.03 1.73 28.44 1.77 42.45 3.47-17.08 9.57-37.02 20.12-58.15 28.4-30.92 12.31-65.47 16.43-99.96 11.91-31.31-3.29-75.69-14.26-109.32-47.85-31.96-31.39-54.2-71.65-62.62-113.36-6.25-38.78-2.22-81.74 11.65-124.26 4.48-12.11 9.84-24.68 18.64-34.59 1.57-1.7 1.54-4.78-1.37-6.79-2.21-1.53-4.32-1.46-6.23.51a91.9 91.9 0 0 0-8.26 10.73c-17.66 28.96-23.69 63.78-26.95 90.95-2.98 28.38-2.07 52.4 2.77 73.44 11.8 47.61 38.27 90.55 74.56 120.9 37.63 31.38 89.56 43.09 138.17 43.09 46.95 0 90.35-19.77 127.51-40.79-3.19 3-28.24 25.35-41.86 35.75-4.49 3.57-2.94 11.76 2.37 11.76s11.49-8.18 15-10.09c9.87-9.95 58.97-43.42 44-57.63l-.2-.16Z"></path></g></g></svg>"""

rotateS ="""<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 500 500"><g fill="#000"><g><path d="M458.38 204.54c-11.8-47.61-38.27-90.55-74.56-120.9-41.46-34.57-96.19-42.5-134.8-43.05-48.75-.98-93.83 19.68-132.19 41.5 10.22-11.39 20.78-22.93 29.95-35.36 2.96-3.91.39-10.74-4.39-10.25-4.84.5-8.13 8.95-11 11.23a879.86 879.86 0 0 1-26.68 31.05c-14.68 15.47-10.31 25.28 11.71 24.94 21.9.45 44.17.37 65.74.46 4.18 0 5.86-6.11 3.12-8.84-2.17-2.16-6.02-1.44-8.5-1.57-18.73-.83-37.72.02-56.43-1.12 16.5-9.21 35.76-19.33 56.38-27.4 30.92-12.31 65.47-16.43 99.96-11.91 31.31 3.29 75.69 14.26 109.32 47.85 31.96 31.39 54.2 71.65 62.62 113.36 6.25 38.78 2.22 81.74-11.65 124.26-12.52 33.82-20.08 32.82-20.07 38.09 0 3.72 5.53 6.99 8.56 3.3a88.7 88.7 0 0 0 8.74-11.25c17.66-28.96 23.69-63.78 26.95-90.95 2.98-28.38 2.07-52.4-2.77-73.44ZM401.17 395.69c-8.77-5.89-20.28-3.78-29.78-5.05-8.61-.9-31.42-1.04-49.14-3.02-2.32-1.28-4.83-.7-6.14 1.11-1.47 2.21-1.79 4.18-.94 5.88 2.84 5.7 17.26 4.45 23.82 5.68 14.03 1.73 28.44 1.77 42.45 3.47-17.08 9.57-37.02 20.12-58.15 28.4-30.92 12.31-65.47 16.43-99.96 11.91-31.31-3.29-75.69-14.26-109.32-47.85-31.96-31.39-54.2-71.65-62.62-113.36-6.25-38.78-2.22-81.74 11.65-124.26 4.48-12.11 9.84-24.68 18.64-34.59 1.57-1.7 1.54-4.78-1.37-6.79-2.21-1.53-4.32-1.46-6.23.51a91.9 91.9 0 0 0-8.26 10.73c-17.66 28.96-23.69 63.78-26.95 90.95-2.98 28.38-2.07 52.4 2.77 73.44 11.8 47.61 38.27 90.55 74.56 120.9 37.63 31.38 89.56 43.09 138.17 43.09 46.95 0 90.35-19.77 127.51-40.79-3.19 3-28.24 25.35-41.86 35.75-4.49 3.57-2.94 11.76 2.37 11.76s11.49-8.18 15-10.09c9.87-9.95 58.97-43.42 44-57.63l-.2-.16Z"></path></g></g></svg>
"""

harvest ="""<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><g fill="#808"><path d="M19.1 4.9C17.3 3.1 14.8 2 12 2 6.5 2 2 6.5 2 12s4.5 10 10 10c2.8 0 5.3-1.1 7.1-2.9L12 12l7.1-7.1z"></path><circle cx="16" cy="12" r="2"></circle><circle cx="21" cy="12" r="2"></circle></g></svg>"""

Absolutely! Those hand-drawn SVGs have much more personality than the programmatic glyphs. Here's how I'd wire them in — wrap each raw SVG string into a sized inline element, and swap them into `_instr_glyph`:

```python
def _svg_icon(raw_svg: str, size=26, flip_x=False, color=None):
    """Wrap a raw SVG string into a fixed-size inline icon.
    
    Args:
        flip_x: mirror horizontally (useful for ROT_L vs ROT_R)
        color: override the fill color (replaces first fill attr)
    """
    svg = raw_svg.strip()
    # Inject sizing attrs
    if 'width=' not in svg[:60]:
        svg = svg.replace('<svg ', f'<svg width="{size}" height="{size}" ', 1)
    else:
        import re
        svg = re.sub(r'width="[^"]*"', f'width="{size}"', svg, count=1)
        svg = re.sub(r'height="[^"]*"', f'height="{size}"', svg, count=1)
    
    if color:
        svg = svg.replace('fill="#000"', f'fill="{color}"', 1)
        svg = svg.replace('fill="#808"', f'fill="{color}"', 1)
    
    style = 'vertical-align:middle;'
    if flip_x:
        style += 'transform:scaleX(-1);'
    
    svg = svg.replace('style="', f'style="{style}', 1) if 'style="' in svg[:200] else \
          svg.replace('<svg ', f'<svg style="{style}" ', 1)
    
    return NotStr(svg)


# Pre-built icons dict
_ICON_SVGS = {}

def _register_icons(arrow_svg, rotate_svg, harvest_svg):
    """Call once after defining the SVG strings to register them."""
    _ICON_SVGS['forward']  = arrow_svg
    _ICON_SVGS['rotate']   = rotate_svg
    _ICON_SVGS['harvest']  = harvest_svg

_register_icons(arrowG, rotateS, harvest)


def _instr_glyph(instr: Instruction, size=26, color="#444"):
    """Inline SVG glyph for one instruction — uses hand-drawn SVGs where available."""
    
    if instr == Instruction.FORWARD and 'forward' in _ICON_SVGS:
        return _svg_icon(_ICON_SVGS['forward'], size, color=color)
    
    if instr == Instruction.ROT_R and 'rotate' in _ICON_SVGS:
        return _svg_icon(_ICON_SVGS['rotate'], size, color=color)
    
    if instr == Instruction.ROT_L and 'rotate' in _ICON_SVGS:
        return _svg_icon(_ICON_SVGS['rotate'], size, flip_x=True, color=color)
    
    if instr == Instruction.HARVEST and 'harvest' in _ICON_SVGS:
        return _svg_icon(_ICON_SVGS['harvest'], size, color=color)
    
    # Fall back to programmatic glyphs for PAUSE, DEFEND, GIVE, SETTLE
    g = DiagramGlyphs(color=color, size=size * 0.55)
    cx, cy = size / 2, size / 2
    body = g.action_glyph(instr, cx, cy) or ""
    return NotStr(f'<svg width="{size}" height="{size}" viewBox="0 0 {size} {size}" '
                  f'style="vertical-align:middle">{body}</svg>')
```

A few things to note:

1. **`rotateS` flipped** via `scaleX(-1)` gives you left vs right rotation — same icon, mirrored
2. **`arrowG`** is actually the same circular arrow as `rotateS` right now — you might want a straight directional arrow for FORWARD instead. If you have one, just register it with `_ICON_SVGS['forward'] = your_straight_arrow`
3. The fallback to `DiagramGlyphs` keeps PAUSE, DEFEND, GIVE, and SETTLE working until you find SVGs for those too

Want to try it and see how it looks, or do you want to find a straight arrow SVG for FORWARD first?

### 4. Piece extensions

In [ ]:
#| export
_PIECE_RENDERERS = {
    PieceType.PAWN:   'pawnPiece',
    PieceType.KNIGHT: 'knightPiece',
    PieceType.ROOK:   'rookPiece',
    PieceType.BISHOP: 'bishopPiece',
    PieceType.QUEEN:  'queenPiece',
    PieceType.KING:   'kingPiece',
}

_PIECE_ICONS = {
    PieceType.PAWN:   "♟", PieceType.KNIGHT: "♞", PieceType.ROOK:   "♜",
    PieceType.BISHOP: "♝", PieceType.QUEEN:  "♛", PieceType.KING:   "♚",
}


@patch
def _ensure_pattern(self: Piece, grid: HexGrid, pat_scale: float = 0.1) -> str:
    """Register flag pattern on grid builder, return CSS fill reference.
    Falls back to flag.primary solid color if no flag."""
    if not self.flag: return "#888"
    pat_name = f"piece_pat_{self.id[:8]}"
    pat = self.flag.flagPattern(pat_name, scale=pat_scale)
    grid.builder.add_definition(pat)
    return f"url(#{pat_name})"


@patch
def draw_svg(self: Piece, grid: HexGrid, hex_idx: int = None,
             scale: float = None, opacity: float = 1.0,
             pat_fill: str = None, suffix: str = "") -> str:
    """Render this piece's chess glyph as SVG at a hex position.

    Args:
        grid:     HexGrid to draw on (needed for hex centres + builder)
        hex_idx:  fine grid index to draw at (defaults to self.location)
        scale:    piece scale (defaults to ~75% of hex diameter)
        opacity:  1.0 for solid, 0.4 for ghost, etc.
        pat_fill: override the fill (otherwise registers flag pattern)
        suffix:   appended to piece_id for uniqueness (e.g. '_ghost')
    """
    idx = hex_idx if hex_idx is not None else self.location
    if idx is None or idx < 0 or idx >= len(grid.hexes): return ""

    center = grid.hexes[idx].center
    renderer = _PIECE_RENDERERS.get(self.piece_type, 'pawnPiece')
    if not self.flag: return ""

    if scale is None:
        scale = (grid.radius * 0.75) / 22.5  # chess SVGs ~22.5px baseline

    fill = pat_fill or self._ensure_pattern(grid)
    pid  = f"piece_{self.id[:8]}{suffix}"

    svg = getattr(self.flag, renderer)(
        MapCord(center.x, center.y),
        scale=scale,
        piece_id=pid,
        fill=fill,
    )

    # Wrap in a group with opacity if not fully opaque
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>\n'

    return svg


@patch
def draw_ghost(self: Piece, grid: HexGrid, hex_idx: int,
               scale: float = None, opacity: float = 0.4) -> str:
    """Draw a faded copy at a projected future position."""
    s = scale or (grid.radius * 0.75) / 22.5
    return self.draw_svg(grid, hex_idx=hex_idx,
                         scale=s * 0.7, opacity=opacity,
                         suffix="_ghost")





In [ ]:
#| export
def StatBar(label, current, maximum, color="auto"):
    """A labeled progress bar for bounded stats like health/food."""
    pct = min(100, max(0, (current / maximum * 100))) if maximum else 0
    if color == "auto":
        color = "success" if pct > 60 else "warning" if pct > 30 else "error"
    return Div(
        DivFullySpaced(
            P(label, cls=TextPresets.muted_sm),
            P(f"{int(current)}/{int(maximum)}", cls="text-xs font-mono"),
        ),
        Progress(value=str(int(pct)), max="100",
                 cls=f"uk-progress h-2 [&::-webkit-progress-value]:bg-{color}"),
        cls="space-y-0.5",
    )

@patch
def __ft__(self: Piece):
    """FastHTML card component for piece detail panel."""
    icon = _PIECE_ICONS.get(self.piece_type, "?")
    ptype = self.piece_type.name.title()
    short_id = self.id[:12] + "…"
    facing_arrow = ["↙","←","↖","↗","→","↘"][self.facing % 6]

    # Progress bars for depletable stats
    bars = [
        StatBar("Health", self.health, self.max_health),
        StatBar("Food", self.food, self.food_capacity),
    ]

    # Numeric stats
    num_stats = [
        ("⚔ Attack",  self.attack_strength),
        ("🏃 Speed",   self.move_strength),
        ("🌾 Harvest", self.harvest_strength),
        ("👁 Sight",   self.sight),
    ]
    stat_grid = Grid(
        *[Div(
            P(label, cls=TextPresets.muted_sm),
            P(str(val), cls="font-bold text-center"),
            cls="text-center",
        ) for label, val in num_stats],
        cols=4,
    )

    return Card(
        CardHeader(
            DivFullySpaced(
                DivLAligned(
                    P(icon, cls="text-3xl"),
                    Div(
                        H4(ptype, cls="font-bold"),
                        P(short_id, cls=TextPresets.muted_sm),
                    ),
                ),
                P(f"Facing {facing_arrow}", cls="text-lg"),
            ),
        ),
        CardBody(
            Div(*bars, cls="space-y-2"),
            Divider(),
            stat_grid,
            Divider(),
            InstructionList(self.rules, self.cursor, self.patrol),
        ),
        CardFooter(
            DivFullySpaced(
                Button("Edit Rules",
                       hx_get=f"/piece_rules/{self.id}",
                       hx_target="#right-panel",
                       cls=ButtonT.primary + " " + ButtonT.sm),
                Button("Show Plan",
                       hx_get=f"/piece_plan/{self.id}",
                       hx_target="#map",
                       cls=ButtonT.sm),
            ),
        ),
        id=f"piece-card-{self.id[:8]}",
    )


#| export
@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_rounds: int = 10, min_elev: float = 1.0,
             countries: np.ndarray = None) -> list['PieceStep']:
    """Walk rules on coarse grid for N steps."""
    if not self.rules or self.location is None: return []
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor

    for n in range(num_rounds):
        if cur >= len(self.rules):
            if self.patrol: cur = 0
            else: break

        instr = Instruction(self.rules[cur])
        prev, blocked = pos, False

        if instr == Instruction.FORWARD:
            d = HexPosition.directions()[fac % 6]
            nbr = grid.hexposition_to_index(d, pos)
            passable = (0 <= nbr < len(elevations)
                        and nbr not in grid.invalidRegion
                        and elevations[nbr] >= min_elev)
            # Also block if country field marks it as water/mountain (-1)
            if passable and countries is not None:
                passable = countries[nbr] >= 0
            if passable:
                pos = nbr
            else:
                blocked = True
        elif instr == Instruction.ROT_L:
            fac = (fac - 1) % 6
        elif instr == Instruction.ROT_R:
            fac = (fac + 1) % 6

        steps.append(PieceStep(hex_idx=pos, facing=fac, instruction=instr,
                               prev_idx=prev, step=n, blocked=blocked))
        cur += 1
    return steps


In [ ]:

@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_turns: int = 10,
             countries: np.ndarray = None,
             occupied: set[int] = None) -> list['PieceStep']:
    """Simulate piece over multiple turns using move_strength as per-turn budget.

    Budget rules:
      - FORWARD: costs _move_cost (1 flat, 2 uphill). If can't afford → end turn.
      - ROT_L / ROT_R: free
      - PAUSE / DEFEND / HARVEST / GIVE / SETTLE: cost 1
      - FORWARD into water/edge: auto-rotate left (free, up to 5×). Full circle → stuck, cost 1.
      - FORWARD into occupied hex: pause instead, cost 1.

    End-of-rules:
      - Last instruction was non-movement → pause out remaining budget.
      - Last instruction was movement/rotate → wrap cursor to 0.
    """
    if not self.rules or self.location is None:
        return []

    occupied = set(occupied) if occupied else set()
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor
    step_n = 0

    for turn in range(num_turns):
        budget = float(self.move_strength)

        while budget > 0:
            # --- End of rules ---
            if cur >= len(self.rules):
                last = Instruction(self.rules[-1])
                if last in _COSTS_ONE:
                    # Idle out the turn
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE,
                                           pos, step_n, turn, budget, 0.0))
                    step_n += 1
                    break
                cur = 0  # wrap around

            instr = Instruction(self.rules[cur])
            prev = pos

            # --- FREE: rotations ---
            if instr in _FREE:
                fac = ((fac - 1) if instr == Instruction.ROT_L
                       else (fac + 1)) % 6
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 0.0, budget))
                step_n += 1; cur += 1
                continue

            # --- COST 1: non-movement actions ---
            if instr in _COSTS_ONE:
                budget -= 1.0
                steps.append(PieceStep(pos, fac, instr, pos,
                                       step_n, turn, 1.0, budget))
                step_n += 1; cur += 1
                continue

            # --- FORWARD ---
            auto_rots = 0
            moved = False
            saved_fac = fac

            while auto_rots <= 5:
                d = HexPosition.directions()[fac % 6]
                nbr = grid.hexposition_to_index(d, pos)

                passable = (0 <= nbr < len(elevations)
                            and nbr not in grid.invalidRegion
                            and elevations[nbr] >= 1)
                if passable and countries is not None:
                    passable = countries[nbr] >= 0

                if not passable:
                    fac = (fac - 1) % 6   # free auto-rotate left
                    auto_rots += 1
                    continue

                # Occupied → forced pause
                if nbr in occupied:
                    steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                           step_n, turn, 1.0, budget - 1.0,
                                           blocked=True,
                                           auto_rotations=auto_rots))
                    budget -= 1.0
                    step_n += 1; cur += 1
                    moved = True
                    break

                # Can we afford it?
                cost = _move_cost(elevations, pos, nbr)
                if cost > budget:
                    budget = 0       # turn over, can't afford
                    moved = True
                    break

                # Move
                occupied.discard(pos)
                pos = nbr
                occupied.add(pos)
                budget -= cost
                steps.append(PieceStep(pos, fac, Instruction.FORWARD, prev,
                                       step_n, turn, cost, budget,
                                       auto_rotations=auto_rots))
                step_n += 1; cur += 1
                moved = True
                break

            if not moved:
                # Spun full circle — stuck
                fac = saved_fac      # 6 left rotations = back to original
                steps.append(PieceStep(pos, fac, Instruction.PAUSE, pos,
                                       step_n, turn, 1.0, budget - 1.0,
                                       blocked=True, auto_rotations=6))
                budget -= 1.0
                step_n += 1; cur += 1

    return steps


def steps_by_turn(steps: list[PieceStep]) -> dict[int, list[PieceStep]]:
    """Group steps by turn number."""
    turns = {}
    for s in steps:
        turns.setdefault(s.turn, []).append(s)
    return turns


@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_turns: int = 10, padding: int = 3,
                countries: np.ndarray = None,
                occupied: set[int] = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_turns,
                          countries=countries, occupied=occupied)
    visited = {self.location}
    for s in steps:
        visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations):
                expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)


#| export
@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_rounds: int = 10, padding: int = 3,
                countries: np.ndarray = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_rounds, countries=countries)
    visited = {self.location}
    for s in steps: visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations): expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)

## Build our testing envoirnment.

### 1. Piece Board

In [ ]:
#| export
@dataclass
class PieceBoard:
    """Lightweight playground for testing piece movement and overlays."""
    grid: HexGrid
    elevations: np.ndarray
    pieces: list  # list[Piece]
    countries: np.ndarray = None
    flags: dict = None  # countryId -> CountryFlag

    @classmethod
    def flat(cls, rings=5, radius=20, num_hexes_elev=200):
        """Open grassland — uniform elevation, no obstacles."""
        style = StyleCSS("pb_hex", fill="#e8e8e0", stroke="#ccc", stroke_width=0.5)
        grid = HexGrid.centered(rings=rings, radius=radius, style=style,
                                center=MapCord(radius*(2*rings+2), radius*(2*rings+2)))
        n = len(grid.hexes)
        elevations = np.full(n, num_hexes_elev, dtype=float)
        countries = np.zeros(n, dtype=float)
        # Mark invalid hexes as water
        for idx in grid.invalidRegion:
            elevations[idx] = 0
            countries[idx] = -1
        return cls(grid=grid, elevations=elevations, pieces=[],
                   countries=countries, flags={})

    @classmethod
    def hilly(cls, rings=5, radius=20, seed=42):
        """Rolling terrain with a river of water hexes through the middle."""
        board = cls.flat(rings=rings, radius=radius)
        rng = np.random.default_rng(seed)
        grid = board.grid
        n = len(grid.hexes)
        # Gentle elevation gradient + noise
        for i in range(n):
            if i in grid.invalidRegion: continue
            pos = grid.index_to_hexposition(i)
            board.elevations[i] = 150 + pos.q * 30 + rng.normal(0, 40)
            board.elevations[i] = max(1, board.elevations[i])
        # Carve a river: zero-elev stripe along r=0
        for i in range(n):
            if i in grid.invalidRegion: continue
            pos = grid.index_to_hexposition(i)
            if abs(pos.r) <= 0 and abs(pos.q) <= rings:
                board.elevations[i] = 0
                board.countries[i] = -1
        return board


In [ ]:
#| export
@patch
def add_piece(self: PieceBoard, hex_idx: int, piece_type=PieceType.PAWN,
              facing: int = 0, rules: list[int] = None,
              country_id: int = 1) -> Piece:
    """Place a new piece on the board."""
    if country_id not in self.flags:
        palette = CountryFlag.seaborn("husl", levels=max(country_id + 2, 4))
        self.flags[country_id] = palette[country_id]
    piece = Piece(
        location=hex_idx, piece_type=piece_type,
        facing=facing, rules=rules or [],
        owner_id=country_id,
    )
    piece.flag = self.flags[country_id]
    self.pieces.append(piece)
    # Paint territory
    if self.countries is not None and self.countries[hex_idx] == 0:
        self.countries[hex_idx] = country_id
    return piece


In [ ]:
#| export
# ── Elevation palette for the playground ─────────────────────────────────────
_PB_WATER  = StyleCSS("pb_water",  fill="#6baed6", stroke="#4292c6", stroke_width=0.5)
_PB_LEVELS = [
    StyleCSS("pb_e0", fill="#c7e9c0", stroke="#bbb", stroke_width=0.5),  # low
    StyleCSS("pb_e1", fill="#a1d99b", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e2", fill="#74c476", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e3", fill="#d4b483", stroke="#bbb", stroke_width=0.5),  # hill
    StyleCSS("pb_e4", fill="#bc8a5f", stroke="#bbb", stroke_width=0.5),
    StyleCSS("pb_e5", fill="#9e6b4a", stroke="#999", stroke_width=0.5),  # peak
]

def _elev_style(elev: float, delta: float = 80.0) -> StyleCSS:
    """Map elevation to a terrain style."""
    if elev < 1: return _PB_WATER
    level = min(len(_PB_LEVELS) - 1, int(elev / delta))
    return _PB_LEVELS[level]


@patch
def _apply_terrain(self: PieceBoard):
    """Paint each hex with its elevation style, registering styles on the builder."""
    grid = self.grid
    for s in [_PB_WATER] + _PB_LEVELS:
        grid.builder.add_style(s)
    for i, hex_obj in enumerate(grid.hexes):
        if i in grid.invalidRegion: continue
        hex_obj.style = _elev_style(self.elevations[i])


@patch
def _pieces_overlay(self: PieceBoard) -> str:
    """SVG string: all pieces at their current locations."""
    svg = ""
    for p in self.pieces:
        svg += p.draw_svg(self.grid)
    return svg






In [ ]:
#| export
@patch
def _plan_overlay(self: PieceBoard, num_rounds: int = 12) -> str:
    """SVG string: simulated movement paths + action glyphs for all pieces."""
    svg = ""
    grid = self.grid

    for p in self.pieces:
        if not p.rules or p.location is None: continue
        steps = p.simulate(grid, self.elevations,
                           num_rounds=num_rounds,
                           countries=self.countries)
        if not steps: continue

        color = p.flag.primary if p.flag else "#333"
        g     = DiagramGlyphs(color=color, size=grid.radius * 0.45)

        # Start ring around piece's current position
        origin = grid.hexes[p.location].center
        svg += g.start_marker(origin.x, origin.y, r=grid.radius * 0.55)

        # Arrow style — dashed, colour-matched to piece
        arrow_style = StyleCSS(
            f"plan_arrow_{p.id[:8]}",
            stroke=color, stroke_width=max(1, grid.radius * 0.08),
            fill="none", opacity="0.7",
            stroke_dasharray=f"{grid.radius*0.25:.0f},{grid.radius*0.12:.0f}",
        )
        grid.builder.add_style(arrow_style)

        prev_idx = p.location
        for step in steps:
            cur = grid.hexes[step.hex_idx].center

            if step.instruction == Instruction.FORWARD and not step.blocked:
                if prev_idx != step.hex_idx:
                    svg += grid.arrow(prev_idx, step.hex_idx,
                                      style=arrow_style, factor=0.3)
                svg += p.draw_ghost(grid, step.hex_idx)

            elif step.blocked:
                svg += g.blocked_x(cur.x, cur.y)

            else:
                svg += g.action_glyph(step.instruction, cur.x, cur.y)

            svg += g.step_label(cur.x, cur.y, step.step)
            prev_idx = step.hex_idx

    return svg


In [ ]:
#| export
@patch
def render(self: PieceBoard, show_plan: bool = False,
           num_rounds: int = 12, width: int = None, height: int = None) -> str:
    """Return a full SVG string of the board."""
    self._apply_terrain()
    grid = self.grid
    grid.update()

    grid.builder.adjust("pieces", self._pieces_overlay())
    grid.builder.adjust("plan", self._plan_overlay(num_rounds) if show_plan else "")

    if width is None or height is None:
        # Derive bounding box from actual hex geometry
        xs = [h.center.x for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        ys = [h.center.y for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        pad = grid.radius * 1.5
        w = int(max(xs) + pad) if width  is None else width
        h = int(max(ys) + pad) if height is None else height
    else:
        w, h = width, height

    grid.builder.width  = w
    grid.builder.height = h
    return grid.builder.xml()


In [ ]:
#| export
@patch
def __ft__(self: PieceBoard):
    """Default view — terrain + pieces, no plan."""
    return Div(NotStr(self.render()))


def PieceBoardPlan(board: PieceBoard, num_rounds: int = 12):
    """View with movement plan overlay."""
    return Div(NotStr(board.render(show_plan=True, num_rounds=num_rounds)))


#### swapping up

In [ ]:
def _turn_end_labels(steps: list[PieceStep]) -> list[tuple[int, int, int]]:
    """Merge consecutive turns that end on the same hex into ranges.
    
    Returns [(hex_idx, first_turn, last_turn), ...] — one entry per
    contiguous run of turns ending at the same hex.
    """
    # Find the hex where each turn ends
    turn_hex = {}  # turn -> hex_idx
    for s in steps:
        turn_hex[s.turn] = s.hex_idx

    if not turn_hex:
        return []

    sorted_turns = sorted(turn_hex)
    runs = []
    run_start = sorted_turns[0]
    prev_turn = run_start
    prev_hex  = turn_hex[run_start]

    for t in sorted_turns[1:]:
        if turn_hex[t] == prev_hex and t == prev_turn + 1:
            prev_turn = t  # extend the run
        else:
            runs.append((prev_hex, run_start, prev_turn))
            run_start = t
            prev_turn = t
            prev_hex  = turn_hex[t]
    runs.append((prev_hex, run_start, prev_turn))
    return runs


In [ ]:
def _range_label(g: DiagramGlyphs, cx, cy, t0, t1, color):
    """Emit '3' for a single turn or '2–4' for a range (1-indexed)."""
    text = str(t0 + 1) if t0 == t1 else f"{t0+1}–{t1+1}"
    offset    = g.size * 0.35
    font_size = max(12, g.size * 0.3)
    return (f'<text x="{cx+offset:.1f}" y="{cy-offset:.1f}" '
            f'text-anchor="start" font-size="{font_size:.0f}" '
            f'fill="{color}" font-family="sans-serif" '
            f'font-weight="bold">{text}</text>\n')


In [ ]:
def piece_plan_overlay(piece: Piece, grid: HexGrid,
                       steps: list[PieceStep] = None,
                       c2f: dict = None,
                       num_turns: int = 12,
                       elevations: np.ndarray = None,
                       countries: np.ndarray = None) -> str:
    if steps is None:
        if elevations is None: return ""
        steps = piece.simulate(grid, elevations, num_turns=num_turns,
                               countries=countries)
    if not steps: return ""

    svg, N = "", len(grid.hexes)
    r = grid.radius
    color = piece.flag.primary if piece.flag else "#333"
    comp  = piece.flag.comp    if piece.flag else "#555"
    g = DiagramGlyphs(color=color, size=r * 0.45)

    arrow_style = StyleCSS(
        f"plan_arr_{piece.id[:8]}",
        stroke=color, stroke_width=max(1, r * 0.08),
        fill="none", opacity="0.7",
        stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}",
    )
    grid.builder.add_style(arrow_style)

    # ── Find last step of each turn ──
    turn_ends = {}
    for i, s in enumerate(steps):
        turn_ends[s.turn] = i  # last one wins

    # ── Start marker ──
    start = _map_point(piece.location, c2f)
    if 0 <= start < N:
        svg += piece.draw_svg(grid, hex_idx=start)
        c = grid.hexes[start].center
        svg += g.start_marker(c.x, c.y, r=r * 0.55)

    # ── Each step ──
    prev_idx = start
    for i, step in enumerate(steps):
        idx = _map_point(step.hex_idx, c2f)
        if idx < 0 or idx >= N:
            prev_idx = idx
            continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        if step.instruction == Instruction.FORWARD and not step.blocked:
            if 0 <= prev_idx < N and prev_idx != idx:
                svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
            svg += piece.draw_ghost(grid, idx)
        elif step.blocked:
            svg += g.blocked_x(cx, cy)
        else:
            svg += g.action_glyph(step.instruction, cx, cy)

        # Only label the last step of each turn
        #if i == turn_ends[step.turn]:
        #    svg += g.step_label(cx, cy, step.turn, color=comp)

        prev_idx = idx

    # ── Range labels (after the step loop) ──
    for hex_idx, t0, t1 in _turn_end_labels(steps):
        idx = _map_point(hex_idx, c2f)
        if 0 <= idx < N:
            cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y
            svg += _range_label(g, cx, cy, t0, t1, comp)


    return svg


In [ ]:
@patch
def _plan_overlay(self: PieceBoard, num_turns: int = 12) -> str:
    """SVG string: simulated movement plan for all pieces."""
    svg = ""
    for p in self.pieces:
        svg += piece_plan_overlay(
            p, self.grid, num_turns=num_turns,
            elevations=self.elevations, countries=self.countries,
        )
    return svg


In [ ]:
@patch
def render(self: PieceBoard, show_plan: bool = False,
           num_turns: int = 12, width: int = None, height: int = None) -> str:
    """Return a full SVG string of the board."""
    self._apply_terrain()
    grid = self.grid
    grid.update()

    grid.builder.adjust("pieces", self._pieces_overlay())
    grid.builder.adjust("plan", self._plan_overlay(num_turns) if show_plan else "")

    if width is None or height is None:
        xs = [h.center.x for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        ys = [h.center.y for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        pad = grid.radius * 1.5
        w = int(max(xs) + pad) if width  is None else width
        h = int(max(ys) + pad) if height is None else height
    else:
        w, h = width, height

    grid.builder.width  = w
    grid.builder.height = h
    return grid.builder.xml()


In [ ]:
@patch
def __ft__(self: PieceBoard):
    """Default view — terrain + pieces, no plan."""
    return Div(NotStr(self.render()))


def PieceBoardPlan(board: PieceBoard, num_turns: int = 12):
    """View with movement plan overlay."""
    return Div(NotStr(board.render(show_plan=True, num_turns=num_turns)))


In [ ]:
#| export
# ── Quick PieceBoard demo ─────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=5, radius=22, seed=7)
grid  = board.grid
mid   = grid.middle

scout = board.add_piece(
    mid, piece_type=PieceType.KNIGHT, facing=3, country_id=1,
    rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
    ],
)

worker = board.add_piece(
    mid - 3, piece_type=PieceType.PAWN, facing=4, country_id=1,
    rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.HARVEST.value, Instruction.HARVEST.value,
        Instruction.ROT_L.value, Instruction.ROT_L.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.GIVE.value,
    ],
)

guard = board.add_piece(
    mid + 7, piece_type=PieceType.ROOK, facing=0, country_id=2,
    rules=[
        Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.DEFEND.value, Instruction.DEFEND.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value,
    ],
)

bishop = board.add_piece(
    mid + grid.nCols * 2, piece_type=PieceType.BISHOP, facing=5, country_id=2,
    rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value,
        Instruction.PAUSE.value,
        Instruction.SETTLE.value,
    ],
)

#show(board)


#show(board)
PieceBoardPlan(board, num_turns=3)

should we be clever if a piece is on a hex for a while we show a range like 2-3

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, that's a nice declutter move! Right now if a piece harvests on the same hex for turns 2, 3, and 4, you'd get three overlapping labels. A range like "2–4" is much cleaner.

The fix is small — instead of labeling every turn-end independently, collect which hex each turn ends on, then merge consecutive runs before emitting labels:

```python
def _turn_end_labels(steps: list[PieceStep]) -> list[tuple[int, int, int]]:
    """Merge consecutive turns that end on the same hex into ranges.
    
    Returns [(hex_idx, first_turn, last_turn), ...] — one entry per
    contiguous run of turns ending at the same hex.
    """
    # Find the hex where each turn ends
    turn_hex = {}  # turn -> hex_idx
    for s in steps:
        turn_hex[s.turn] = s.hex_idx

    if not turn_hex:
        return []

    sorted_turns = sorted(turn_hex)
    runs = []
    run_start = sorted_turns[0]
    prev_turn = run_start
    prev_hex  = turn_hex[run_start]

    for t in sorted_turns[1:]:
        if turn_hex[t] == prev_hex and t == prev_turn + 1:
            prev_turn = t  # extend the run
        else:
            runs.append((prev_hex, run_start, prev_turn))
            run_start = t
            prev_turn = t
            prev_hex  = turn_hex[t]
    runs.append((prev_hex, run_start, prev_turn))
    return runs
```

Then in `piece_plan_overlay`, replace the per-step labeling with a single pass at the end:

```python
def _range_label(g: DiagramGlyphs, cx, cy, t0, t1, color):
    """Emit '3' for a single turn or '2–4' for a range (1-indexed)."""
    text = str(t0 + 1) if t0 == t1 else f"{t0+1}–{t1+1}"
    offset    = g.size * 0.35
    font_size = max(12, g.size * 0.3)
    return (f'<text x="{cx+offset:.1f}" y="{cy-offset:.1f}" '
            f'text-anchor="start" font-size="{font_size:.0f}" '
            f'fill="{color}" font-family="sans-serif" '
            f'font-weight="bold">{text}</text>\n')
```

And the label section in the overlay becomes:

```python
    # ── Range labels (after the step loop) ──
    for hex_idx, t0, t1 in _turn_end_labels(steps):
        idx = _map_point(hex_idx, c2f)
        if 0 <= idx < N:
            cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y
            svg += _range_label(g, cx, cy, t0, t1, comp)
```

…and remove the inline `step_label` call from inside the step loop entirely.

This way a pawn that parks and harvests for 3 turns shows a single tidy **"2–4"** badge instead of three stacked numbers fighting for the same pixel.

Anything else you would change?A few things I'd tighten up, roughly in priority order:

**1. Ghost IDs aren't unique** — every ghost gets `id="piece_66ff270b_ghost"`, so you end up with duplicate IDs in the SVG. Some browsers get confused. Quick fix in `draw_ghost` / `draw_svg`:

```python
suffix=f"_ghost_{step_n}"  # or pass step index through
```

**2. Pattern defs are registered once per draw call** — look at the output: `piece_pat_66ff270b` appears as a `<pattern>` 15+ times. `_ensure_pattern` creates a new object each call; `add_definition` deduplicates by `id` but still runs every time. A one-liner cache would help:

```python
@patch
def _ensure_pattern(self: Piece, grid: HexGrid, pat_scale: float = 0.1) -> str:
    if not self.flag: return "#888"
    pat_name = f"piece_pat_{self.id[:8]}"
    if pat_name not in grid.builder.styles:  # or track in a set
        pat = self.flag.flagPattern(pat_name, scale=pat_scale)
        grid.builder.add_definition(pat)
    return f"url(#{pat_name})"
```

**3. `occupied` isn't threaded between pieces** — `_plan_overlay` calls `piece_plan_overlay` per piece but doesn't accumulate an `occupied` set across them. So two pieces can happily walk through each other in the preview. If you want collision-aware plans, build the set once and pass it through in chess-rank order.

**4. Budget can go negative** — in the "blocked" and "stuck" branches, `budget -= 1.0` fires without checking `budget >= 1.0`. If a piece has 0.5 budget left and gets blocked, `budget_remaining` becomes –0.5. A `budget = max(0, budget - 1.0)` guard would be cleaner.

**5. Food/diet isn't deducted yet** — the design says deduct `diet` at turn end and lose health when `food < 0`, but `simulate` doesn't touch `food` or `health`. Not a bug today since it's all read-only simulation, but worth a `# TODO` so you remember when you wire up actual round resolution.

The first two are the most immediately impactful — they affect every render right now.

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

#| export
@patch
def create_game(self: GameStorage, user_id, template_name="bayArea",
                kingdoms=5, lakes=1, radius=10) -> ActiveGame:
    logging.info(f"create_game: user={user_id} template={template_name}")
    tt = TerrainTemplate()
    terrain = getattr(tt, template_name)()
    
    terrain.carve_to_ocean(num_lakes=lakes)
    terrain.hexGrid.adjustRadius(radius)

    # GameBoard now creates cover + basins internally
    board = GameBoard(terrain, top_n=kingdoms)
    board.expand_kingdoms(max_rounds=50)
    
    # Save using the cover that GameBoard created
    board.cover.db = self
    board.cover.save(name=template_name)
    board.save(self, world_id=board.cover.ident)

    self.db.execute("UPDATE user SET activeWorld = ? WHERE id = ?",
                    [board.cover.ident, user_id])
    
    return ActiveGame(board=board, cover=board.cover, world_id=board.cover.ident)


_game_cache = {}
_cache_lock = threading.Lock()

@patch
def active_board(self: GameStorage, user_id) -> ActiveGame:
    with _cache_lock:
        if user_id in _game_cache:
            logging.info(f"active_board: cache hit for user {user_id}")
            return _game_cache[user_id]
    
    logging.info(f"active_board: cache miss, building for user {user_id}")
    row = self.db.execute("SELECT activeWorld FROM user WHERE id = ?", [user_id]).fetchone()
    if not row or not row[0]:
        return None
    
    world_id = row[0]
    try:
        # gameboard() now loads cover internally and reuses cover.basin
        board = self.gameboard(world_id)
        active = ActiveGame(board=board, cover=board.cover, world_id=world_id)
        
        with _cache_lock:
            _game_cache[user_id] = active
        
        return active
    except Exception as e:
        logging.error(f"active_board: FAILED: {e}", exc_info=True)
        return None


In [ ]:
def piece_plan_overlay(piece: Piece, grid: HexGrid,
                       steps: list[PieceStep] = None,
                       c2f: dict = None,
                       num_turns: int = 12,
                       elevations: np.ndarray = None,
                       countries: np.ndarray = None) -> str:
    """Render movement plan overlay for a single piece.

    Pass pre-computed `steps`, or provide `elevations` to simulate on the fly.
    """
    if steps is None:
        if elevations is None:
            return ""
        steps = piece.simulate(grid, elevations, num_turns=num_turns,
                               countries=countries)
    if not steps:
        return ""

    svg, N = "", len(grid.hexes)
    r = grid.radius
    color = piece.flag.primary if piece.flag else "#333"
    comp  = piece.flag.comp    if piece.flag else "#555"
    g = DiagramGlyphs(color=color, size=r * 0.45)

    # Dashed arrow style for movement lines
    arrow_style = StyleCSS(
        f"plan_arr_{piece.id[:8]}",
        stroke=color, stroke_width=max(1, r * 0.08),
        fill="none", opacity="0.7",
        stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}",
    )
    grid.builder.add_style(arrow_style)

    # ── Start: full piece + ring ──
    start = _map_point(piece.location, c2f)
    if 0 <= start < N:
        svg += piece.draw_svg(grid, hex_idx=start)
        c = grid.hexes[start].center
        svg += g.start_marker(c.x, c.y, r=r * 0.55)

    # ── Each step ──
    prev_idx = start
    for step in steps:
        idx = _map_point(step.hex_idx, c2f)
        if idx < 0 or idx >= N:
            prev_idx = idx
            continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        if step.instruction == Instruction.FORWARD and not step.blocked:
            if 0 <= prev_idx < N and prev_idx != idx:
                svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
            svg += piece.draw_ghost(grid, idx)

        elif step.blocked:
            svg += g.blocked_x(cx, cy)

        else:
            svg += g.action_glyph(step.instruction, cx, cy)

        svg += g.step_label(cx, cy, step.step, color=comp)
        prev_idx = idx

    return svg


#### some junk

@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    logging.info(f"create world redirecting")
    return RedirectResponse('/game', status_code=303)


@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    logging.info(f"create_world: session keys={list(session.keys())}")
    invalidate_cache(uid)  # <-- clear old game
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    return RedirectResponse('/game', status_code=303)

In [ ]:
# Empty before we go

In [ ]:
@rt("/piece_detail/{id}")
def piece_detail(session, id: str):
    """Placeholder for piece detail — right panel."""
    return Div(
        H4(f"🧑 Piece", cls="font-bold"),
        P(f"ID: {id[:12]}…", cls=TextPresets.muted_sm),
        Divider(),
        P("Actions and stats coming soon", cls="opacity-50 text-sm"),
        cls="space-y-2"
    )


## Editors

### 1. Piece Drag editor

In [ ]:
def piece_drag_editor(piece: Piece, rings: int = 3, radius: int = 35):
    """HexDragMap centered on piece location.
    Drag a path → POST hex_ids → convert to rules → display compact badge strip.
    """
    base    = StyleCSS("pe-hex",   fill="#1e293b", stroke="#475569", stroke_width=1)
    current = StyleCSS("pe-cur",   fill="#f59e0b", stroke="#fbbf24", stroke_width=2)
    lbl     = StyleCSS("pe-label", fill="#e2e8f0", font_size="11px")

    grid = HexGrid.centered(rings=rings, radius=radius, style=base)
    for s in [current, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle

    # Mark piece's home hex
    grid.hexes[mid].style     = current
    grid.hexes[mid].label     = piece.piece_type.value[0].upper()
    grid.hexes[mid].labelStyle = lbl.name

    # Facing arrow from centre hex
    facing_dir = HexPosition.directions()[piece.facing % 6]
    facing_idx = grid.hexposition_to_index(facing_dir, mid)
    if facing_idx >= 0:
        grid.builder.adjust("facing", grid.arrow(
            mid, facing_idx,
            style=StyleCSS("pe-arrow", stroke="#f59e0b", stroke_width=2),
        ))

    grid.update()

    return Div(
        H5("Drag a patrol path", cls="text-sm font-bold mb-1"),
        P("Drag across hexes to generate movement instructions",
          cls="text-xs opacity-50 mb-2"),
        HexDragMap(
            grid,
            on_drag={
                "hx_post":   f"/piece_path/{piece.id}/{piece.facing}",
                "hx_target": f"#rules-{piece.id[:8]}",
                "hx_swap":   "outerHTML",
            },
            id=f"drag-editor-{piece.id[:8]}",
        ),
        InstructionList(piece.rules, piece.cursor, piece.patrol).compact(piece.id),
        cls="space-y-2",
    )


In [ ]:
@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request: Request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return compact badge strip."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # Reconstruct the same grid used in piece_drag_editor
    rings, radius = 3, 35
    grid = HexGrid.centered(rings=rings, radius=radius,
                            style=StyleCSS("x", fill="white"))

    new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist — append to existing rules, or replace if you prefer
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.get("rules") or "").split(",") if x]
            new_rules = existing + new_rules
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in new_rules)},
                pk="id"
            )
    except Exception:
        pass  # demo mode — show without saving

    return InstructionList(new_rules).compact(piece_id)


Do we have enough to make an editable list of instructions? We want to be able to delete and pick an insertion point where we can use the drag extension

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! We have all the pieces — `InstructionList.__ft__` gives us the table, `piece_drag_editor` gives us the drag input, and the HTMX plumbing is in place. We need to wire them together with delete buttons and an insertion cursor.

The shape I'm thinking:

```
┌─ Instructions ───────────────────────────────────┐
│  #   glyph   action         [insert here] [del]  │
│  1   →       Forward            ○           ✕    │
│  ▶2  ↺       Turn Left          ○           ✕    │  ← cursor (insert point)
│  3   🌾      Harvest             ○           ✕    │
│              [+ Add via drag]                     │
│  🔁 Patrol                                        │
└──────────────────────────────────────────────────┘
```

The `○` button sets the cursor (insert point), `✕` deletes that row, and the drag editor inserts *at* the cursor. Three HTMX endpoints:

```python
@rt("/piece_rules/{piece_id}")
def piece_rules(piece_id: str):
    """Full rules editor panel — table + drag editor."""
    ...

@rt("/piece_rule_delete/{piece_id}/{idx:int}")
def piece_rule_delete(piece_id: str, idx: int):
    """Delete instruction at idx, return updated editor."""
    ...

@rt("/piece_rule_cursor/{piece_id}/{idx:int}")  
def piece_rule_cursor(piece_id: str, idx: int):
    """Move insert cursor to idx, return updated editor."""
    ...
```

And the drag endpoint already exists but needs to insert at cursor rather than append.

Want me to write it all out? The key question first: where does the piece state live during editing — purely in the DB, or do you want an in-memory edit buffer so changes only commit when the user hits **Apply**?

Lets do the apply version. We might want to do heavy lifting on the back end to render

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call — keeps the DB clean and lets us do all the SVG simulation rendering server-side on Apply. The pattern is:

- **Edit buffer** — `dict[piece_id, EditSession]` in memory, keyed by piece_id
- **All mutations** (delete, cursor move, drag insert) update the buffer and return a fresh re-render of the whole editor panel
- **Apply** — writes buffer to DB, triggers a map re-render via OOB swap
- **Cancel** — drops the buffer

```python
@dataclass
class RuleEditSession:
    """In-memory edit buffer for a piece's rules."""
    piece_id: str
    rules:    list[int]
    cursor:   int = 0       # insert point
    patrol:   bool = True

_rule_sessions: dict[str, RuleEditSession] = {}


def _get_session(piece_id: str) -> RuleEditSession | None:
    return _rule_sessions.get(piece_id)

def _open_session(piece: Piece) -> RuleEditSession:
    sess = RuleEditSession(
        piece_id=piece.id,
        rules=list(piece.rules),
        cursor=piece.cursor,
        patrol=piece.patrol,
    )
    _rule_sessions[piece.id] = sess
    return sess
```

Then the editor component that re-renders itself completely on every mutation:

```python
def _rules_editor(sess: RuleEditSession, grid: HexGrid) -> FT:
    """Full server-rendered rules editor panel."""
    pid  = sess.piece_id[:8]
    rows = []
    for i, r in enumerate(sess.rules):
        instr        = Instruction(r)
        label, _     = INSTR_LABELS.get(instr, (instr.name, "?"))
        is_cursor    = (i == sess.cursor)

        cursor_btn = Button(
            "▶" if is_cursor else "○",
            hx_post=f"/piece_rule_cursor/{sess.piece_id}/{i}",
            hx_target=f"#rule-editor-{pid}",
            hx_swap="outerHTML",
            cls=(ButtonT.primary if is_cursor else ButtonT.ghost) + " " + ButtonT.xs,
            title="Set insert point here",
        )
        delete_btn = Button(
            UkIcon("x", height=12),
            hx_post=f"/piece_rule_delete/{sess.piece_id}/{i}",
            hx_target=f"#rule-editor-{pid}",
            hx_swap="outerHTML",
            cls=ButtonT.destructive + " " + ButtonT.xs,
        )
        rows.append([
            cursor_btn,
            _instr_glyph(instr),
            label,
            delete_btn,
        ])

    tbl = TableFromLists(["", "", "Action", ""], rows,
                         cls=(TableT.sm, TableT.striped))

    patrol_toggle = Div(
        Label(
            Switch(
                checked=sess.patrol,
                hx_post=f"/piece_rule_patrol/{sess.piece_id}",
                hx_target=f"#rule-editor-{pid}",
                hx_swap="outerHTML",
            ),
            "Patrol (loop)",
        ),
        cls="flex items-center gap-2 mt-2",
    )

    action_row = DivFullySpaced(
        Button("Apply",
               hx_post=f"/piece_rule_apply/{sess.piece_id}",
               hx_target=f"#rule-editor-{pid}",
               hx_swap="outerHTML",
               hx_vals='{"render": "1"}',
               cls=ButtonT.primary + " " + ButtonT.sm),
        Button("Cancel",
               hx_post=f"/piece_rule_cancel/{sess.piece_id}",
               hx_target=f"#rule-editor-{pid}",
               hx_swap="outerHTML",
               cls=ButtonT.ghost + " " + ButtonT.sm),
        cls="mt-2",
    )

    return Card(
        H5("Edit Rules", cls="font-bold mb-2"),
        tbl if sess.rules else P("No instructions yet", cls="opacity-50 text-sm"),
        patrol_toggle,
        Divider(),
        piece_drag_editor_inline(sess, grid),   # drag input at cursor
        Divider(),
        action_row,
        id=f"rule-editor-{pid}",
    )
```

The drag editor knows the cursor so it inserts at the right point:

```python
def piece_drag_editor_inline(sess: RuleEditSession, grid: HexGrid) -> FT:
    """Compact drag strip — inserts at session cursor."""
    pid = sess.piece_id[:8]
    return Div(
        P(f"Drag to insert at position {sess.cursor}",
          cls="text-xs opacity-50"),
        HexDragMap(
            grid,
            on_drag={
                "hx_post":   f"/piece_path/{sess.piece_id}/{0}",
                "hx_target": f"#rule-editor-{pid}",
                "hx_swap":   "outerHTML",
            },
        ),
    )
```

The five endpoints are all one-liners once you have the session:

```python
@rt("/piece_rules/{piece_id}")
def piece_rules(session, piece_id: str):
    """Open editor — load piece from DB, create session."""
    piece = globalStore.piece_from_id(piece_id)
    sess  = _open_session(piece)
    rings, radius = 3, 35
    grid  = HexGrid.centered(rings=rings, radius=radius,
                             style=StyleCSS("x", fill="#1e293b"))
    return _rules_editor(sess, grid)


@rt("/piece_rule_cursor/{piece_id}/{idx:int}")
def piece_rule_cursor(piece_id: str, idx: int):
    sess = _get_session(piece_id)
    if sess: sess.cursor = max(0, min(idx, len(sess.rules)))
    grid = HexGrid.centered(rings=3, radius=35, style=StyleCSS("x", fill="#1e293b"))
    return _rules_editor(sess, grid)


@rt("/piece_rule_delete/{piece_id}/{idx:int}")
def piece_rule_delete(piece_id: str, idx: int):
    sess = _get_session(piece_id)
    if sess and 0 <= idx < len(sess.rules):
        sess.rules.pop(idx)
        sess.cursor = min(sess.cursor, len(sess.rules))
    grid = HexGrid.centered(rings=3, radius=35, style=StyleCSS("x", fill="#1e293b"))
    return _rules_editor(sess, grid)


@rt("/piece_rule_patrol/{piece_id}")
def piece_rule_patrol(piece_id: str):
    sess = _get_session(piece_id)
    if sess: sess.patrol = not sess.patrol
    grid = HexGrid.centered(rings=3, radius=35, style=StyleCSS("x", fill="#1e293b"))
    return _rules_editor(sess, grid)


@rt("/piece_rule_apply/{piece_id}")
def piece_rule_apply(piece_id: str):
    sess = _get_session(piece_id)
    if sess:
        # Persist to DB
        globalStore.pieces.upsert({
            "id":     piece_id,
            "rules":  ",".join(str(r) for r in sess.rules),
            "cursor": sess.cursor,
            "patrol": 1 if sess.patrol else 0,
        }, pk="id")
        del _rule_sessions[piece_id]

    # Re-render the plan SVG server-side as an OOB swap
    piece  = globalStore.piece_from_id(piece_id)
    board  = PieceBoard.flat(rings=5, radius=22)
    board.pieces = [piece]
    plan_svg = NotStr(board.render(show_plan=True, num_rounds=12))

    return (
        P("Saved ✓", cls="text-sm text-green-500"),          # replaces editor
        Div(plan_svg, id="map", hx_swap_oob="innerHTML"),     # updates map OOB
    )


@rt("/piece_rule_cancel/{piece_id}")
def piece_rule_cancel(piece_id: str):
    _rule_sessions.pop(piece_id, None)
    return P("Cancelled", cls="text-sm opacity-50")
```

The drag path endpoint inserts at cursor rather than appending:

```python
@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request: Request, piece_id: str, facing: int):
    form    = await request.form()
    raw     = form.get("hex_ids", "")
    hex_ids = [int(x) for x in raw.split(",") if x.strip()]
    if not hex_ids: return P("No path", cls="opacity-50 text-sm")

    grid      = HexGrid.centered(rings=3, radius=35, style=StyleCSS("x", fill="white"))
    new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)

    sess = _get_session(piece_id)
    if sess:
        # Insert at cursor position
        for i, r in enumerate(new_rules):
            sess.rules.insert(sess.cursor + i, r)
        sess.cursor += len(new_rules)

    return _rules_editor(sess, grid)
```

The heavy Apply rendering happens entirely server-side — the client just receives a finished SVG blob. Want to wire this into the main game panel next?

Should RuleEditSession use an InstructionList? should we combine classes?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good instinct — they overlap almost completely. But I'd say **compose, don't combine**. They have different jobs:

- **`InstructionList`** — the *model*: holds rules/cursor/patrol, renders them, converts paths. It's a value object you can pass around, serialize, stick in a `Piece.__ft__`.
- **`RuleEditSession`** — the *session envelope*: adds `piece_id` for cache lookup, and exists only while the editor is open. It's ephemeral server state.

So `RuleEditSession` becomes a thin wrapper:

```python
@dataclass
class RuleEditSession:
    """In-memory edit buffer for a piece's rules."""
    piece_id: str
    instructions: InstructionList

_rule_sessions: dict[str, RuleEditSession] = {}

def _open_session(piece: Piece) -> RuleEditSession:
    sess = RuleEditSession(
        piece_id=piece.id,
        instructions=InstructionList(list(piece.rules), piece.cursor, piece.patrol),
    )
    _rule_sessions[piece.id] = sess
    return sess
```

Then add mutation methods to `InstructionList` so the endpoints stay clean:

```python
@patch
def delete_at(self: InstructionList, idx: int):
    """Remove instruction at idx, clamping cursor."""
    if 0 <= idx < len(self.rules):
        self.rules.pop(idx)
        self.cursor = min(self.cursor, len(self.rules))

@patch
def insert_rules(self: InstructionList, new_rules: list[int]):
    """Insert rules at cursor position, advance cursor past them."""
    for i, r in enumerate(new_rules):
        self.rules.insert(self.cursor + i, r)
    self.cursor += len(new_rules)

@patch
def set_cursor(self: InstructionList, idx: int):
    self.cursor = max(0, min(idx, len(self.rules)))

@patch
def toggle_patrol(self: InstructionList):
    self.patrol = not self.patrol
```

Now every endpoint is a one-liner on `sess.instructions`:

```python
# Delete
sess.instructions.delete_at(idx)

# Cursor move
sess.instructions.set_cursor(idx)

# Drag insert
new_rules = InstructionList.path_to_rules(hex_ids, grid, facing)
sess.instructions.insert_rules(new_rules)

# Patrol toggle
sess.instructions.toggle_patrol()

# Apply → write back to piece
piece.rules  = sess.instructions.rules
piece.cursor = sess.instructions.cursor
piece.patrol = sess.instructions.patrol
```

The nice thing is `InstructionList` stays useful *outside* the editor — `Piece.__ft__` already uses it for the readonly display, `compact()` works in summaries, `from_path()` works standalone. Adding mutation methods doesn't break any of that. The session just gives it an address in the server cache.

### 2. Rule Edit Session

In [ ]:
@dataclass
class RuleEditSession:
    """In-memory edit buffer for a piece's rules."""
    piece_id: str
    instructions: InstructionList

_rule_sessions: dict[str, RuleEditSession] = {}

def _open_session(piece: Piece) -> RuleEditSession:
    sess = RuleEditSession(
        piece_id=piece.id,
        instructions=InstructionList(list(piece.rules), piece.cursor, piece.patrol),
    )
    _rule_sessions[piece.id] = sess
    return sess


In [ ]:
def _auth_piece(session, piece_id: str) -> tuple[RuleEditSession, int] | None:
    """Return (session, user_id) if the user owns this piece, else None.
    
    Checks:
      1. User is logged in (ensure_user)
      2. Piece belongs to the user's active world
      3. Piece's kingdom matches one the user controls
      4. An edit session exists (for mutation routes)
    """
    uid    = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return None

    # Does this piece belong to the active world?
    try:
        rec = globalStore.pieces[piece_id]
    except (KeyError, NotFoundError):
        return None

    r = rec if isinstance(rec, dict) else asdict(rec)
    if r.get("world_id") != active.world_id:
        return None

    # Does the piece's kingdom belong to this user's game?
    piece_kingdom = r.get("kingdom_id", 0)
    valid_kingdoms = {k.countryId for k in active.board.kingdoms}
    if piece_kingdom not in valid_kingdoms:
        return None

    return uid


def _auth_session(session, piece_id: str) -> RuleEditSession | None:
    """Like _auth_piece but also requires an active edit session."""
    uid = _auth_piece(session, piece_id)
    if uid is None:
        return None
    return _rule_sessions.get(piece_id)


_DENIED = P("⛔ Not authorized", cls="text-sm text-red-500")
_EXPIRED = P("Session expired — reopen the editor", cls="text-sm text-red-500")


In [ ]:
# ── Shared grid factory ──────────────────────────────────────────────────────
def _editor_grid(rings=3, radius=35):
    return HexGrid.centered(
        rings=rings, radius=radius,
        style=StyleCSS("pe-hex", fill="#1e293b", stroke="#475569", stroke_width=1),
    )

def _editor_id(piece_id): return f"rule-editor-{piece_id[:8]}"


# ── Editor panel component ───────────────────────────────────────────────────
def _rules_editor(sess: RuleEditSession) -> FT:
    """Full server-rendered rules editor. Re-rendered on every mutation."""
    eid  = _editor_id(sess.piece_id)
    il   = sess.instructions
    rows = []

    for i, r in enumerate(il.rules):
        instr     = Instruction(r)
        label, _  = INSTR_LABELS.get(instr, (instr.name, "?"))
        is_cursor = (i == il.cursor)

        cursor_btn = Button(
            "▶" if is_cursor else "○",
            hx_post=piece_rule_cursor.to(piece_id=sess.piece_id, idx=i),
            hx_target=f"#{eid}", hx_swap="outerHTML",
            cls=(ButtonT.primary if is_cursor else ButtonT.ghost) + " " + ButtonT.xs,
        )
        delete_btn = Button(
            UkIcon("x", height=12),
            hx_post=piece_rule_delete.to(piece_id=sess.piece_id, idx=i),
            hx_target=f"#{eid}", hx_swap="outerHTML",
            hx_confirm="Delete this instruction?",
            cls=ButtonT.destructive + " " + ButtonT.xs,
        )
        rows.append([cursor_btn, _instr_glyph(instr), label, delete_btn])

    # "Append at end" cursor slot
    at_end = il.cursor >= len(il.rules)
    rows.append([
        Button("▶" if at_end else "○",
               hx_post=piece_rule_cursor.to(piece_id=sess.piece_id, idx=len(il.rules)),
               hx_target=f"#{eid}", hx_swap="outerHTML",
               cls=(ButtonT.primary if at_end else ButtonT.ghost) + " " + ButtonT.xs),
        "", "— end —", "",
    ])

    tbl = TableFromLists(["", "", "Action", ""], rows,
                         cls=(TableT.sm, TableT.striped))

    patrol_toggle = LabelCheckboxX(
        "Patrol (loop)", checked=il.patrol,
        hx_post=piece_rule_patrol.to(piece_id=sess.piece_id),
        hx_target=f"#{eid}", hx_swap="outerHTML",
    )

    # Drag editor inserts at cursor
    grid = _editor_grid()
    grid.update()
    drag_section = Div(
        P(f"Drag to insert at position {il.cursor + 1}",
          cls="text-xs opacity-60 mb-1"),
        HexDragMap(grid, on_drag={
            "hx_post":   piece_path.to(piece_id=sess.piece_id, facing=0),
            "hx_target": f"#{eid}",
            "hx_swap":   "outerHTML",
        }),
    )

    action_row = DivFullySpaced(
        Button("Apply", cls=ButtonT.primary + " " + ButtonT.sm,
               hx_post=piece_rule_apply.to(piece_id=sess.piece_id),
               hx_target=f"#{eid}", hx_swap="outerHTML",
               hx_indicator=f"#{eid}"),
        Button("Cancel", cls=ButtonT.ghost + " " + ButtonT.sm,
               hx_post=piece_rule_cancel.to(piece_id=sess.piece_id),
               hx_target=f"#{eid}", hx_swap="outerHTML"),
    )

    return Card(
        H5("Edit Rules", cls="font-bold"),
        tbl if il.rules else P("No instructions — drag a path below",
                                cls="opacity-50 text-sm"),
        patrol_toggle,
        Divider(),
        drag_section,
        Divider(),
        action_row,
        id=eid, cls="htmx-indicator",
    )




In [ ]:
@rt
def piece_rules(session, piece_id: str):
    """Open rules editor from current DB state."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    piece = globalStore.piece_from_id(piece_id)
    return _rules_editor(_open_session(piece))

@rt
def piece_rule_cursor(session, piece_id: str, idx: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.set_cursor(idx)
    return _rules_editor(sess)

@rt
def piece_rule_delete(session, piece_id: str, idx: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.delete_at(idx)
    return _rules_editor(sess)

@rt
def piece_rule_patrol(session, piece_id: str):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.toggle_patrol()
    return _rules_editor(sess)

@rt
async def piece_path(session, request, piece_id: str, facing: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    form    = await request.form()
    raw     = form.get("hex_ids", "")
    hex_ids = [int(x) for x in raw.split(",") if x.strip()]
    if hex_ids:
        grid = _editor_grid()
        new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)
        sess.instructions.insert_rules(new_rules)
    return _rules_editor(sess)

@rt
def piece_rule_apply(session, piece_id: str):
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    sess = _rule_sessions.pop(piece_id, None)
    if not sess: return _EXPIRED

    il = sess.instructions
    globalStore.pieces.upsert({
        "id": piece_id,
        "rules": ",".join(str(r) for r in il.rules),
        "cursor": il.cursor,
        "patrol": 1 if il.patrol else 0,
    }, pk="id")

    eid = _editor_id(piece_id)
    editor_result = Div(
        P("✓ Rules saved", cls="text-sm text-green-600 font-bold mb-2"),
        il.compact(piece_id),
        id=eid,
    )
    map_update = Div(P("Map updated", cls="text-xs opacity-50"),
                     id="map", hx_swap_oob="innerHTML")
    return editor_result, map_update

@rt
def piece_rule_cancel(session, piece_id: str):
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    sess = _rule_sessions.pop(piece_id, None)
    eid  = _editor_id(piece_id)
    if sess:
        return Div(sess.instructions.compact(piece_id), id=eid)
    return _EXPIRED


So we want to make sure that the user has permission to edit these routes

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right — every one of those routes is wide open right now. The cleanest pattern is a small guard function that each route calls, so you tighten it in one place:

```python
def _auth_piece(session, piece_id: str) -> tuple[RuleEditSession, int] | None:
    """Return (session, user_id) if the user owns this piece, else None.
    
    Checks:
      1. User is logged in (ensure_user)
      2. Piece belongs to the user's active world
      3. Piece's kingdom matches one the user controls
      4. An edit session exists (for mutation routes)
    """
    uid    = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return None

    # Does this piece belong to the active world?
    try:
        rec = globalStore.pieces[piece_id]
    except (KeyError, NotFoundError):
        return None

    r = rec if isinstance(rec, dict) else asdict(rec)
    if r.get("world_id") != active.world_id:
        return None

    # Does the piece's kingdom belong to this user's game?
    piece_kingdom = r.get("kingdom_id", 0)
    valid_kingdoms = {k.countryId for k in active.board.kingdoms}
    if piece_kingdom not in valid_kingdoms:
        return None

    return uid


def _auth_session(session, piece_id: str) -> RuleEditSession | None:
    """Like _auth_piece but also requires an active edit session."""
    uid = _auth_piece(session, piece_id)
    if uid is None:
        return None
    return _rule_sessions.get(piece_id)


_DENIED = P("⛔ Not authorized", cls="text-sm text-red-500")
_EXPIRED = P("Session expired — reopen the editor", cls="text-sm text-red-500")
```

Then every route becomes guarded with one line at the top:

```python
@rt
def piece_rules(session, piece_id: str):
    """Open rules editor from current DB state."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    piece = globalStore.piece_from_id(piece_id)
    return _rules_editor(_open_session(piece))

@rt
def piece_rule_cursor(session, piece_id: str, idx: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.set_cursor(idx)
    return _rules_editor(sess)

@rt
def piece_rule_delete(session, piece_id: str, idx: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.delete_at(idx)
    return _rules_editor(sess)

@rt
def piece_rule_patrol(session, piece_id: str):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    sess.instructions.toggle_patrol()
    return _rules_editor(sess)

@rt
async def piece_path(session, request, piece_id: str, facing: int):
    sess = _auth_session(session, piece_id)
    if not sess: return _EXPIRED
    form    = await request.form()
    raw     = form.get("hex_ids", "")
    hex_ids = [int(x) for x in raw.split(",") if x.strip()]
    if hex_ids:
        grid = _editor_grid()
        new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)
        sess.instructions.insert_rules(new_rules)
    return _rules_editor(sess)

@rt
def piece_rule_apply(session, piece_id: str):
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    sess = _rule_sessions.pop(piece_id, None)
    if not sess: return _EXPIRED

    il = sess.instructions
    globalStore.pieces.upsert({
        "id": piece_id,
        "rules": ",".join(str(r) for r in il.rules),
        "cursor": il.cursor,
        "patrol": 1 if il.patrol else 0,
    }, pk="id")

    eid = _editor_id(piece_id)
    editor_result = Div(
        P("✓ Rules saved", cls="text-sm text-green-600 font-bold mb-2"),
        il.compact(piece_id),
        id=eid,
    )
    map_update = Div(P("Map updated", cls="text-xs opacity-50"),
                     id="map", hx_swap_oob="innerHTML")
    return editor_result, map_update

@rt
def piece_rule_cancel(session, piece_id: str):
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED
    sess = _rule_sessions.pop(piece_id, None)
    eid  = _editor_id(piece_id)
    if sess:
        return Div(sess.instructions.compact(piece_id), id=eid)
    return _EXPIRED
```

The key checks:
1. **`ensure_user`** — are they logged in at all?
2. **`world_id` match** — does this piece live in their active game?
3. **`kingdom_id` match** — does the piece belong to a kingdom in that game?
4. **Session exists** — for mutation routes, was the editor actually opened?

Later when you add multiplayer (each player controls one kingdom), you'd tighten check 3 to `piece_kingdom == active.country_id` instead of checking all kingdoms.

Should this use The hx-delete attribute in htmx? issues an HTTP DELETE request to a specified URL, typically to remove a resource on the server. It is used in combination with other htmx attributes like hx-target and hx-swap to define how the page content updates after the request. 

and others. We want this to be as friendly in the eco HTMLX and FASTHTML system as possible

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question. The FastHTML docs are actually explicit about this:

> "We recommend generally sticking to just these two HTTP verbs [GET and POST]."

So `hx_post` for mutations is idiomatic FastHTML. **But** there are several HTMX/FastHTML patterns we're *not* using that would make this much cleaner:

**1. Route function references** — no f-strings for URLs:
```python
# Instead of: hx_post=f"/piece_rule_delete/{sess.piece_id}/{i}"
# Use:        hx_post=piece_rule_delete.to(piece_id=sess.piece_id, idx=i)
```

**2. `hx_confirm` on destructive actions**

**3. `hx_swap_oob` on Apply** — update the map *and* the editor in one response

**4. `hx_indicator`** — show loading during heavy SVG rendering

Here's the cleaned-up version:

```python
# ── Shared grid factory ──────────────────────────────────────────────────────
def _editor_grid(rings=3, radius=35):
    return HexGrid.centered(
        rings=rings, radius=radius,
        style=StyleCSS("pe-hex", fill="#1e293b", stroke="#475569", stroke_width=1),
    )

def _editor_id(piece_id): return f"rule-editor-{piece_id[:8]}"


# ── Editor panel component ───────────────────────────────────────────────────
def _rules_editor(sess: RuleEditSession) -> FT:
    """Full server-rendered rules editor. Re-rendered on every mutation."""
    eid  = _editor_id(sess.piece_id)
    il   = sess.instructions
    rows = []

    for i, r in enumerate(il.rules):
        instr     = Instruction(r)
        label, _  = INSTR_LABELS.get(instr, (instr.name, "?"))
        is_cursor = (i == il.cursor)

        cursor_btn = Button(
            "▶" if is_cursor else "○",
            hx_post=piece_rule_cursor.to(piece_id=sess.piece_id, idx=i),
            hx_target=f"#{eid}", hx_swap="outerHTML",
            cls=(ButtonT.primary if is_cursor else ButtonT.ghost) + " " + ButtonT.xs,
        )
        delete_btn = Button(
            UkIcon("x", height=12),
            hx_post=piece_rule_delete.to(piece_id=sess.piece_id, idx=i),
            hx_target=f"#{eid}", hx_swap="outerHTML",
            hx_confirm="Delete this instruction?",
            cls=ButtonT.destructive + " " + ButtonT.xs,
        )
        rows.append([cursor_btn, _instr_glyph(instr), label, delete_btn])

    # "Append at end" cursor slot
    at_end = il.cursor >= len(il.rules)
    rows.append([
        Button("▶" if at_end else "○",
               hx_post=piece_rule_cursor.to(piece_id=sess.piece_id, idx=len(il.rules)),
               hx_target=f"#{eid}", hx_swap="outerHTML",
               cls=(ButtonT.primary if at_end else ButtonT.ghost) + " " + ButtonT.xs),
        "", "— end —", "",
    ])

    tbl = TableFromLists(["", "", "Action", ""], rows,
                         cls=(TableT.sm, TableT.striped))

    patrol_toggle = LabelCheckboxX(
        "Patrol (loop)", checked=il.patrol,
        hx_post=piece_rule_patrol.to(piece_id=sess.piece_id),
        hx_target=f"#{eid}", hx_swap="outerHTML",
    )

    # Drag editor inserts at cursor
    grid = _editor_grid()
    grid.update()
    drag_section = Div(
        P(f"Drag to insert at position {il.cursor + 1}",
          cls="text-xs opacity-60 mb-1"),
        HexDragMap(grid, on_drag={
            "hx_post":   piece_path.to(piece_id=sess.piece_id, facing=0),
            "hx_target": f"#{eid}",
            "hx_swap":   "outerHTML",
        }),
    )

    action_row = DivFullySpaced(
        Button("Apply", cls=ButtonT.primary + " " + ButtonT.sm,
               hx_post=piece_rule_apply.to(piece_id=sess.piece_id),
               hx_target=f"#{eid}", hx_swap="outerHTML",
               hx_indicator=f"#{eid}"),
        Button("Cancel", cls=ButtonT.ghost + " " + ButtonT.sm,
               hx_post=piece_rule_cancel.to(piece_id=sess.piece_id),
               hx_target=f"#{eid}", hx_swap="outerHTML"),
    )

    return Card(
        H5("Edit Rules", cls="font-bold"),
        tbl if il.rules else P("No instructions — drag a path below",
                                cls="opacity-50 text-sm"),
        patrol_toggle,
        Divider(),
        drag_section,
        Divider(),
        action_row,
        id=eid, cls="htmx-indicator",
    )


# ── Routes ───────────────────────────────────────────────────────────────────

@rt
def piece_rules(session, piece_id: str):
    """Open rules editor from current DB state."""
    piece = globalStore.piece_from_id(piece_id)
    return _rules_editor(_open_session(piece))

@rt
def piece_rule_cursor(piece_id: str, idx: int):
    sess = _rule_sessions.get(piece_id)
    if not sess: return P("Session expired", cls="text-sm text-red-500")
    sess.instructions.set_cursor(idx)
    return _rules_editor(sess)

@rt
def piece_rule_delete(piece_id: str, idx: int):
    sess = _rule_sessions.get(piece_id)
    if not sess: return P("Session expired", cls="text-sm text-red-500")
    sess.instructions.delete_at(idx)
    return _rules_editor(sess)

@rt
def piece_rule_patrol(piece_id: str):
    sess = _rule_sessions.get(piece_id)
    if not sess: return P("Session expired", cls="text-sm text-red-500")
    sess.instructions.toggle_patrol()
    return _rules_editor(sess)

@rt
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex path, insert as rules at cursor."""
    form    = await request.form()
    raw     = form.get("hex_ids", "")
    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    sess = _rule_sessions.get(piece_id)
    if not sess: return P("Session expired", cls="text-sm text-red-500")

    if hex_ids:
        grid = _editor_grid()
        new_rules = InstructionList.path_to_rules(hex_ids, grid, start_facing=facing)
        sess.instructions.insert_rules(new_rules)

    return _rules_editor(sess)

@rt
def piece_rule_apply(piece_id: str):
    """Commit edits to DB, render plan SVG as OOB swap."""
    sess = _rule_sessions.pop(piece_id, None)
    if not sess: return P("Session expired", cls="text-sm text-red-500")

    il = sess.instructions
    globalStore.pieces.upsert({
        "id":     piece_id,
        "rules":  ",".join(str(r) for r in il.rules),
        "cursor": il.cursor,
        "patrol": 1 if il.patrol else 0,
    }, pk="id")

    eid = _editor_id(piece_id)

    # Editor slot: show confirmation + compact summary
    editor_result = Div(
        P("✓ Rules saved", cls="text-sm text-green-600 font-bold mb-2"),
        il.compact(piece_id),
        id=eid,
    )

    # OOB: re-render map with updated plan
    # (heavy SVG work happens here, server-side)
    # TODO: wire to real ActiveGame board once integrated
    map_update = Div(
        P("Map updated", cls="text-xs opacity-50"),
        id="map", hx_swap_oob="innerHTML",
    )

    return editor_result, map_update

@rt
def piece_rule_cancel(piece_id: str):
    """Discard edits, tear down session."""
    sess = _rule_sessions.pop(piece_id, None)
    eid  = _editor_id(piece_id)
    if sess:
        return Div(il.compact(piece_id), id=eid)
    return P("Session expired", cls="text-sm text-red-500", id=eid)
```

The key improvements:
- **`route_fn.to()`** everywhere — no f-string URL construction, FastHTML fills path params
- **`@rt` with no path arg** — function name *is* the route, idiomatic FastHTML
- **`hx_confirm`** on delete — native HTMX confirm dialog, zero JS
- **`hx_indicator`** on Apply — the card gets the `htmx-request` class during the heavy render, so you can add an opacity/spinner transition with CSS
- **OOB swap** on Apply — updates both the editor panel *and* the map in a single response via `hx_swap_oob="innerHTML"`
- **Tuple return** from Apply — FastHTML concatenates both partials, HTMX processes each one (main target + OOB)

### 3. Direction Picker

In [ ]:
#| export
_DIR_ARROWS = ["↙", "←", "↖", "↗", "→", "↘"]   # keep

@rt
def set_facing(session, piece_id: str, facing: int):
    """Update piece facing — works with edit session if open."""
    uid = _auth_piece(session, piece_id)
    if uid is None: return _DENIED

    sess = _rule_sessions.get(piece_id)
    if sess:
        # If editor is open, update session (uncommitted)
        # facing lives on the Piece, not InstructionList — 
        # so we'd need to stash it. For now just re-render compass.
        pass

    facing = facing % 6
    target_id = f"facing-{piece_id[:8]}"

    # Reload piece, update, re-render compass
    piece = globalStore.piece_from_id(piece_id)
    piece.facing = facing
    globalStore.pieces.upsert({"id": piece_id, "facing": facing}, pk="id")

    return facing_grid(piece)


In [ ]:
def facing_grid(piece: Piece, radius: int = 30):
    """Compass rose: piece glyph in center, arrows pointing to each direction."""
    piece_id = piece.id
    base   = StyleCSS("fc-hex",    fill="#f8f8f8", stroke="#999", stroke_width=1.5)
    active = StyleCSS("fc-active", fill="#4CAF50", stroke="#333", stroke_width=2)
    center = StyleCSS("fc-center", fill="#e0e0e0", stroke="#666", stroke_width=1.5)

    grid = HexGrid.centered(1, radius=radius, style=base)
    for s in [active, center]:
        grid.builder.add_style(s)

    mid = grid.middle

    # Direction hexes
    dir_map = {}
    for fi, d in enumerate(HexPosition.directions()):
        gi = grid.hexposition_to_index(d, mid)
        dir_map[gi] = fi
        grid.hexes[gi].style = active if fi == piece.facing else base
        grid.hexes[gi].label = _DIR_ARROWS[fi]
        grid.hexes[gi].labelStyle = "fc-label"

    grid.hexes[mid].style = center
    grid.builder.add_style(StyleCSS("fc-label", font_size="13px"))
    grid.update()

    # Overlay: piece in center + arrows from center to each neighbor
    overlay = piece.draw_svg(grid, hex_idx=mid, scale=(radius * 0.6) / 22.5)

    arrow_active = StyleCSS("fc-arr-on",  stroke="#4CAF50", stroke_width=2, fill="none")
    arrow_dim    = StyleCSS("fc-arr-off", stroke="#bbb",    stroke_width=1, fill="none")
    grid.builder.add_style(arrow_active)
    grid.builder.add_style(arrow_dim)

    for gi, fi in dir_map.items():
        style = arrow_active if fi == piece.facing else arrow_dim
        overlay += grid.arrow(mid, gi, style=style, factor=0.35)

    grid.builder.adjust("overlay", overlay)

    target_id = f"facing-{piece_id[:8]}"

    def on_click(g, i):
        if i in dir_map:
            return {
                "hx-post":   f"/set_facing/{piece_id}/{dir_map[i]}",
                "hx-target": f"#{target_id}",
                "hx-swap":   "outerHTML",
            }
        return {}

    return Div(
        H5("Facing", cls="font-bold text-sm mb-1"),
        HexTouchMap(grid, on_click=on_click),
        id=target_id,
        cls="inline-block",
    )


## Getting Data

In [ ]:
showUsers()

In [ ]:
dummySession= {'userid': 64801 }

In [ ]:
#globalStore.create_game(dummySession['userid'], template_name="bayArea")

Can we add this dummy user and then add a Knight piece to the first settlement in the first kingom of its active world

In [ ]:
def addMissing():
    from datetime import datetime

    # 1. Add the dummy user
    now = int(datetime.now().timestamp())
    globalStore.users.insert(dict(
        username="dummy", email="dummy@test.com", password="test",
        created=now, sessionID="", activeWorld=0, id=64801
    ))

    # 2. Create a world for them
    active = globalStore.create_game(64801, template_name="bayArea")

    # 3. Recruit a Knight into the first settlement of the first kingdom
    board = active.board
    kingdom = board.kingdoms[0]
    settlement = kingdom.settlements[0]
    settlement.db = globalStore  # wire up DB so save works

    knight = settlement.recruit(PieceType.KNIGHT, flag=kingdom.flag)
    knight.save(db=globalStore, world_id=active.world_id, kingdom_id=kingdom.countryId)

    print(f"User: 64801")
    print(f"World: {active.world_id}")
    print(f"Kingdom: {kingdom.countryName} (id={kingdom.countryId})")
    print(f"Settlement: {settlement.name} @ hex {settlement.location}")
    print(f"Knight: {knight.id[:12]}… facing={knight.facing} atk={knight.attack_strength} mov={knight.move_strength}")


In [ ]:
#addMissing()

In [ ]:
def showPieces():
    users_df = pd.DataFrame(globalStore.pieces())
    print("pieces:")
    print(users_df)

In [ ]:
showPieces()

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret


In [ ]:
showActiveSettlements(dummySession)

In [ ]:
def listSettlements():
    users_df = pd.DataFrame(globalStore.settlements())
    print("pieces:")
    print(users_df)

In [ ]:
listSettlements()

In [ ]:
dummySettlment = "7ba2cc4b-97fd-4682-8d34-e4b9ea06cd72"

In [ ]:
@rt("/piece_map/{id}")
def piece_map(session, id: str, rings: int = None):
    """Map partial — renders zoomed terrain with active overlays."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game", id="map")

    if rings is not None:
        rings = max(3, min(12, rings))
        session['settlement_rings'] = rings
    else:
        rings = session.get('settlement_rings', 5)

    overlays = get_overlays(session)

    try:
        place = globalStore.piece_from_id(id)
        board = active.board
        terrain = board.terrain

        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty", id="map")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()
        c2f = result.invert_mapper()

        piece_attrs = {
            'hx-get': f'/piece_detail/{id}',
            'hx-target': '#right-panel',
            'hx-swap': 'innerHTML',
            'style': 'cursor: pointer;',
        }
        zoomed.apply_overlays(result, board, c2f, overlays, piece_attrs)

        return Div(
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )
    except Exception as e:
        logging.error(f"settlement_map FAILED: {e}", exc_info=True)
        return P(f"Error: {e}", id="map")


Piece needs to have an overlay. lets start building the one.

So I am going to start having the flag be what marks settlments so the king returns (good book). So the idea is for a piece map that would show what the piece would do. we should have a number of rounds to show where the piece thinks is going to happen based upon its rules. We are going to want to have that for a group of pieces as well when we have the settlement and kindom maps. We need a region large enough to show all of the moves. so we will have to find the hexes in the fine region and then use the zoom-region to get them which is a smidge complicated

I guess we need a few parts on the middle where we get a large patch of terrain because we have to see if the piece deadends in water and has to turn to figure out the hexes. then use that as the basis for the region. The second time we should be hitting a cache. Though it might be easier to just get the elevations instead of the full result.

so we are going to want a datastructue to draw the piece future movement 
def pieceMove
    startIndex
    endIndex
    cost # in case we need to draw a swiggle
    stopKind = harvest, give, defend.

    but this is really the rules list anyways. we just need to append the approaite svg to a str

Do you think you can build simulare, the overlay and a demo?

#| export
@dataclass
class PieceStep:
    """One resolved step in a piece's movement plan."""
    hex_idx: int
    facing: int
    instruction: Instruction
    prev_idx: int
    step: int
    blocked: bool = False




#| export
@patch
def simulate(self: Piece, grid: HexGrid, elevations: np.ndarray,
             num_rounds: int = 10, min_elev: float = 1.0,
             countries: np.ndarray = None) -> list['PieceStep']:
    """Walk rules on coarse grid for N steps."""
    if not self.rules or self.location is None: return []
    steps = []
    pos, fac, cur = self.location, self.facing, self.cursor

    for n in range(num_rounds):
        if cur >= len(self.rules):
            if self.patrol: cur = 0
            else: break

        instr = Instruction(self.rules[cur])
        prev, blocked = pos, False

        if instr == Instruction.FORWARD:
            d = HexPosition.directions()[fac % 6]
            nbr = grid.hexposition_to_index(d, pos)
            passable = (0 <= nbr < len(elevations)
                        and nbr not in grid.invalidRegion
                        and elevations[nbr] >= min_elev)
            # Also block if country field marks it as water/mountain (-1)
            if passable and countries is not None:
                passable = countries[nbr] >= 0
            if passable:
                pos = nbr
            else:
                blocked = True
        elif instr == Instruction.ROT_L:
            fac = (fac - 1) % 6
        elif instr == Instruction.ROT_R:
            fac = (fac + 1) % 6

        steps.append(PieceStep(hex_idx=pos, facing=fac, instruction=instr,
                               prev_idx=prev, step=n, blocked=blocked))
        cur += 1
    return steps

@patch
def plan_region(self: Piece, grid: HexGrid, elevations: np.ndarray,
                num_rounds: int = 10, padding: int = 3,
                countries: np.ndarray = None
                ) -> tuple[list['PieceStep'], 'HexRegion']:
    """Simulate and return (steps, bounding_region) for zooming."""
    steps = self.simulate(grid, elevations, num_rounds, countries=countries)
    visited = {self.location}
    for s in steps: visited.add(s.hex_idx)
    expanded = set()
    for h in visited:
        for n in grid.indices_in_range(h, padding):
            if 0 <= n < len(elevations): expanded.add(n)
    return steps, HexRegion(hexes=expanded, hexGrid=grid)


def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       color: str = "#4CAF50", c2f: dict = None,
                       start_idx: int = -1) -> str:
    """Render plan as SVG: dashed arrows for moves, glyphs for actions."""
    svg, N = "", len(grid.hexes)

    # Start ring
    si = _map_pt(start_idx, c2f) if start_idx >= 0 else -1
    if 0 <= si < N:
        sx, sy = grid.hexes[si].center.x, grid.hexes[si].center.y
        svg += (f'\t<circle cx="{sx:.1f}" cy="{sy:.1f}" r="7" '
                f'fill="none" stroke="{color}" stroke-width="2.5" opacity="0.9"/>\n')

    for step in steps:
        idx  = _map_pt(step.hex_idx, c2f)
        prev = _map_pt(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if idx < 0 or idx >= N: continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        # Movement line + dot
        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= prev < N and prev != idx):
            px, py = grid.hexes[prev].center.x, grid.hexes[prev].center.y
            svg += (f'\t<line x1="{px:.1f}" y1="{py:.1f}" x2="{cx:.1f}" y2="{cy:.1f}" '
                    f'stroke="{color}" stroke-width="2" '
                    f'stroke-dasharray="4,2" opacity="0.7"/>\n')
            svg += (f'\t<circle cx="{cx:.1f}" cy="{cy:.1f}" r="3.5" '
                    f'fill="{color}" opacity="0.6"/>\n')

        # Blocked
        elif step.blocked:
            svg += (f'\t<circle cx="{cx:.1f}" cy="{cy:.1f}" r="5" '
                    f'fill="red" stroke="darkred" stroke-width="1" opacity="0.5"/>\n')

        # Action glyph
        else:
            glyph = _STEP_GLYPHS.get(step.instruction, "")
            if glyph:
                svg += (f'\t<text x="{cx:.1f}" y="{cy:.1f}" text-anchor="middle" '
                        f'dominant-baseline="middle" font-size="13">{glyph}</text>\n')

        # Step number
        svg += (f'\t<text x="{cx+7:.1f}" y="{cy-7:.1f}" text-anchor="start" '
                f'font-size="8" fill="#555" font-weight="bold">{step.step+1}</text>\n')

    return svg

_PIECE_RENDERERS = {
    PieceType.PAWN:   'pawnPiece',
    PieceType.KNIGHT: 'knightPiece',
    PieceType.ROOK:   'rookPiece',
    PieceType.BISHOP: 'bishopPiece',
    PieceType.QUEEN:  'queenPiece',
    PieceType.KING:   'kingPiece',
}

_STEP_GLYPHS = {
    Instruction.PAUSE:   "⏸",
    Instruction.DEFEND:  "🛡",
    Instruction.HARVEST: "🌾",
    Instruction.GIVE:    "🤝",
    Instruction.SETTLE:  "🏠",
    Instruction.ROT_L:   "↺",
    Instruction.ROT_R:   "↻",
}

def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       piece: Piece = None, c2f: dict = None,
                       color: str = "#4CAF50", move_scale: float = 1.5,
                       piece_scale: float = 0.6) -> str:
    """Render movement plan using grid arrows and flag chess pieces."""
    svg, N = "", len(grid.hexes)
    flag = piece.flag if piece else None

    # Arrow style for movement lines
    arrow_style = StyleCSS(f"plan_arrow_{color.strip('#')}",
                           stroke=color, stroke_width=2,
                           fill="none", opacity="0.7",
                           stroke_dasharray="6,3")
    grid.builder.add_style(arrow_style)

    # Draw the piece SVG at its start location
    start_fine = _map_point(piece.location, c2f) if piece and piece.location is not None else -1
    if 0 <= start_fine < N and flag:
        center = grid.hexes[start_fine].center
        renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
        svg += getattr(flag, renderer)(
            MapCord(center.x, center.y),
            scale=piece_scale,
            piece_id=f"plan_start_{piece.id[:8]}",
        )

    for step in steps:
        fine_idx  = _map_point(step.hex_idx, c2f)
        fine_prev = _map_point(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if fine_idx < 0 or fine_idx >= N:
            continue
        cx, cy = grid.hexes[fine_idx].center.x, grid.hexes[fine_idx].center.y

        # === FORWARD move → spline arrow ===
        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= fine_prev < N and fine_prev != fine_idx):
            svg += grid.arrow(fine_prev, fine_idx, style=arrow_style, factor=0.3)

        # === Blocked move → red X ===
        elif step.blocked:
            svg += (f'\t<circle cx="{cx:.1f}" cy="{cy:.1f}" r="5" '
                    f'fill="red" stroke="darkred" stroke-width="1" opacity="0.5"/>\n')
            svg += (f'\t<text x="{cx:.1f}" y="{cy+1:.1f}" text-anchor="middle" '
                    f'dominant-baseline="middle" font-size="9" fill="white" '
                    f'font-weight="bold">✘</text>\n')

        # === Action glyph (harvest, defend, etc.) ===
        else:
            glyph = _STEP_GLYPHS.get(step.instruction, "")
            if glyph:
                svg += (f'\t<text x="{cx:.1f}" y="{cy:.1f}" text-anchor="middle" '
                        f'dominant-baseline="middle" font-size="14">{glyph}</text>\n')

        # Step number label (offset up-right)
        svg += (f'\t<text x="{cx+8:.1f}" y="{cy-8:.1f}" text-anchor="start" '
                f'font-size="8" fill="#555" font-weight="bold">{step.step+1}</text>\n')

    # Ghost piece at final position (smaller, semi-transparent)
    if steps and flag:
        last = steps[-1]
        final_fine = _map_point(last.hex_idx, c2f)
        if 0 <= final_fine < N and final_fine != start_fine:
            end_center = grid.hexes[final_fine].center
            renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
            svg += getattr(flag, renderer)(
                MapCord(end_center.x, end_center.y),
                scale=piece_scale * 0.7,
                piece_id=f"plan_end_{piece.id[:8]}",
                attrs={"opacity": "0.4"},
            )

    return svg


_STEP_GLYPHS = {
    Instruction.PAUSE:   "⏸",
    Instruction.DEFEND:  "🛡",
    Instruction.HARVEST: "🌾",
    Instruction.GIVE:    "🤝",
    Instruction.SETTLE:  "🏠",
    Instruction.ROT_L:   "↺",
    Instruction.ROT_R:   "↻",
}

def _map_pt(coarse_idx, c2f=None):
    """Map coarse index to fine (or identity if no c2f)."""
    if c2f is None: return coarse_idx
    fs = c2f.get(coarse_idx)
    return fs[len(fs)//2] if fs else -1




### Knight draw demo

hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

# Background rects so the flag pattern shows behind the king
for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName,scale=0.1)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    # Draw a colored circle as backdrop, then the king on top
    king_scale = 3.0
    backdrop = f'<circle cx="{cx}" cy="{cy}" r="{radius * 0.7}" fill="white" stroke="black" stoke-width="1"/>'
    pat_fill = f"url(#{patternName})"
    king_svg = flag.knightPiece(MapCord(cx, cy), scale=king_scale, piece_id=f"king_{i}", fill=pat_fill)

    
    canvas.adjust(f"backdrop-{i}", backdrop)
    canvas.adjust(f"king-{i}", king_svg)
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="sans-serif" fill="#333">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


def svg_circle(cx, cy, r, fill="none", stroke="#333", stroke_width=1.5, opacity=1.0, **kw):
    """Filled or outlined circle."""
    return (f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{r:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}"/>\n')

def svg_ring(cx, cy, r, stroke, stroke_width=2, opacity=0.9):
    """Hollow ring (start marker, selection, etc.)."""
    return svg_circle(cx, cy, r, fill="none", stroke=stroke,
                      stroke_width=stroke_width, opacity=opacity)

def svg_arc(cx, cy, r, start_deg, end_deg, stroke, stroke_width=2, opacity=0.8):
    """Arc segment — good for partial rings, rotation indicators."""
    import math
    s = math.radians(start_deg)
    e = math.radians(end_deg)
    x1, y1 = cx + r * math.cos(s), cy + r * math.sin(s)
    x2, y2 = cx + r * math.cos(e), cy + r * math.sin(e)
    large = 1 if (end_deg - start_deg) % 360 > 180 else 0
    return (f'<path d="M {x1:.1f},{y1:.1f} A {r:.1f},{r:.1f} 0 {large} 1 {x2:.1f},{y2:.1f}" '
            f'fill="none" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}" stroke-linecap="round"/>\n')

def svg_shield(cx, cy, size, fill, stroke, opacity=0.85):
    """Shield shape for DEFEND — classic heraldic proportions."""
    w, h = size, size * 1.2
    return (f'<path d="M {cx:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy+h*0.1:.1f} '
            f'Q {cx:.1f},{cy+h*0.5:.1f} {cx-w*0.5:.1f},{cy+h*0.1:.1f} '
            f'L {cx-w*0.5:.1f},{cy-h*0.5:.1f} Z" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')

def svg_sheaf(cx, cy, size, fill, stroke, opacity=0.85):
    """Wheat sheaf for HARVEST — three lines bound in middle."""
    s = size
    svg = ""
    # Three stalks
    for dx in [-s*0.3, 0, s*0.3]:
        svg += (f'<line x1="{cx+dx:.1f}" y1="{cy-s*0.5:.1f}" '
                f'x2="{cx+dx*0.3:.1f}" y2="{cy+s*0.5:.1f}" '
                f'stroke="{fill}" stroke-width="{max(1.5, s*0.15):.1f}" '
                f'stroke-linecap="round" opacity="{opacity}"/>\n')
    # Binding band
    svg += (f'<line x1="{cx-s*0.35:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.35:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="{max(1, s*0.1):.1f}" opacity="{opacity}"/>\n')
    return svg

def svg_gift(cx, cy, size, fill, stroke, opacity=0.85):
    """Simple box with ribbon for GIVE."""
    s = size * 0.8
    svg = (f'<rect x="{cx-s*0.5:.1f}" y="{cy-s*0.4:.1f}" '
           f'width="{s:.1f}" height="{s*0.8:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'rx="{s*0.1:.1f}" opacity="{opacity}"/>\n')
    # Ribbon cross
    svg += (f'<line x1="{cx:.1f}" y1="{cy-s*0.4:.1f}" '
            f'x2="{cx:.1f}" y2="{cy+s*0.4:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.5:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.5:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg

def svg_house(cx, cy, size, fill, stroke, opacity=0.85):
    """Simple house for SETTLE."""
    s = size
    # Walls
    svg = (f'<rect x="{cx-s*0.4:.1f}" y="{cy:.1f}" '
           f'width="{s*0.8:.1f}" height="{s*0.5:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    # Roof
    svg += (f'<polygon points="'
            f'{cx-s*0.5:.1f},{cy:.1f} '
            f'{cx:.1f},{cy-s*0.45:.1f} '
            f'{cx+s*0.5:.1f},{cy:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg

def svg_pause(cx, cy, size, fill, stroke, opacity=0.7):
    """Two vertical bars for PAUSE."""
    w, h = size * 0.2, size * 0.6
    for dx in [-size*0.18, size*0.18]:
        fill_c = (f'<rect x="{cx+dx-w/2:.1f}" y="{cy-h/2:.1f}" '
                  f'width="{w:.1f}" height="{h:.1f}" '
                  f'fill="{fill}" stroke="{stroke}" stroke-width="1" '
                  f'rx="{w*0.2:.1f}" opacity="{opacity}"/>\n')
        yield fill_c

def svg_rot_arrow(cx, cy, r, clockwise: bool, stroke, stroke_width=2, opacity=0.85):
    """Circular arc arrow for ROT_L / ROT_R."""
    # 270° arc + arrowhead
    if clockwise:
        arc = svg_arc(cx, cy, r, -120, 150, stroke, stroke_width, opacity)
        # Arrowhead tip at end_deg=150
        import math
        tip_x = cx + r * math.cos(math.radians(150))
        tip_y = cy + r * math.sin(math.radians(150))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x-r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x-r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    else:
        arc = svg_arc(cx, cy, r, 30, 300, stroke, stroke_width, opacity)
        import math
        tip_x = cx + r * math.cos(math.radians(30))
        tip_y = cy + r * math.sin(math.radians(30))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x+r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x+r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    return arc + ah


Lets migrate all of these to be @patched on a DigramGlyphs

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's the class with all the helpers patched on. I kept a few shared defaults on `self` so you don't repeat them everywhere:

```python
@dataclass
class DiagramGlyphs:
    """Shared SVG glyph renderer for piece-plan overlays."""
    color:        str   = "#333"
    stroke_width: float = 1.5
    opacity:      float = 0.85
    size:         float = 10.0


@patch
def circle(self: DiagramGlyphs, cx, cy, r, fill="none", stroke=None,
           stroke_width=None, opacity=None, **kw):
    stroke = stroke or self.color
    stroke_width = stroke_width or self.stroke_width
    opacity = opacity if opacity is not None else self.opacity
    return (f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{r:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{stroke_width}" '
            f'opacity="{opacity}"/>\n')


@patch
def ring(self: DiagramGlyphs, cx, cy, r, stroke=None, stroke_width=2, opacity=0.9):
    return self.circle(cx, cy, r, fill="none",
                       stroke=stroke or self.color,
                       stroke_width=stroke_width, opacity=opacity)


@patch
def arc(self: DiagramGlyphs, cx, cy, r, start_deg, end_deg,
        stroke=None, stroke_width=2, opacity=0.8):
    import math
    stroke = stroke or self.color
    s, e = math.radians(start_deg), math.radians(end_deg)
    x1, y1 = cx + r * math.cos(s), cy + r * math.sin(s)
    x2, y2 = cx + r * math.cos(e), cy + r * math.sin(e)
    large = 1 if (end_deg - start_deg) % 360 > 180 else 0
    return (f'<path d="M {x1:.1f},{y1:.1f} A {r:.1f},{r:.1f} 0 {large} 1 '
            f'{x2:.1f},{y2:.1f}" fill="none" stroke="{stroke}" '
            f'stroke-width="{stroke_width}" opacity="{opacity}" '
            f'stroke-linecap="round"/>\n')


@patch
def shield(self: DiagramGlyphs, cx, cy, size=None, fill=None,
           stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    w, h = size, size * 1.2
    return (f'<path d="M {cx:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy-h*0.5:.1f} '
            f'L {cx+w*0.5:.1f},{cy+h*0.1:.1f} '
            f'Q {cx:.1f},{cy+h*0.5:.1f} {cx-w*0.5:.1f},{cy+h*0.1:.1f} '
            f'L {cx-w*0.5:.1f},{cy-h*0.5:.1f} Z" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
            f'opacity="{opacity}"/>\n')


@patch
def sheaf(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size
    svg = ""
    for dx in [-s*0.3, 0, s*0.3]:
        svg += (f'<line x1="{cx+dx:.1f}" y1="{cy-s*0.5:.1f}" '
                f'x2="{cx+dx*0.3:.1f}" y2="{cy+s*0.5:.1f}" '
                f'stroke="{fill}" stroke-width="{max(1.5, s*0.15):.1f}" '
                f'stroke-linecap="round" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.35:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.35:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="{max(1, s*0.1):.1f}" '
            f'opacity="{opacity}"/>\n')
    return svg


@patch
def gift(self: DiagramGlyphs, cx, cy, size=None, fill=None,
         stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size * 0.8
    svg = (f'<rect x="{cx-s*0.5:.1f}" y="{cy-s*0.4:.1f}" '
           f'width="{s:.1f}" height="{s*0.8:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'rx="{s*0.1:.1f}" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx:.1f}" y1="{cy-s*0.4:.1f}" '
            f'x2="{cx:.1f}" y2="{cy+s*0.4:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    svg += (f'<line x1="{cx-s*0.5:.1f}" y1="{cy:.1f}" '
            f'x2="{cx+s*0.5:.1f}" y2="{cy:.1f}" '
            f'stroke="{stroke}" stroke-width="1.5" opacity="{opacity}"/>\n')
    return svg


@patch
def house(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=None):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    s = size
    svg = (f'<rect x="{cx-s*0.4:.1f}" y="{cy:.1f}" '
           f'width="{s*0.8:.1f}" height="{s*0.5:.1f}" '
           f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
           f'opacity="{opacity}"/>\n')
    svg += (f'<polygon points="'
            f'{cx-s*0.5:.1f},{cy:.1f} '
            f'{cx:.1f},{cy-s*0.45:.1f} '
            f'{cx+s*0.5:.1f},{cy:.1f}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5" '
            f'opacity="{opacity}"/>\n')
    return svg


@patch
def pause(self: DiagramGlyphs, cx, cy, size=None, fill=None,
          stroke=None, opacity=0.7):
    size = size or self.size
    fill = fill or self.color
    stroke = stroke or self.color
    w, h = size * 0.2, size * 0.6
    svg = ""
    for dx in [-size*0.18, size*0.18]:
        svg += (f'<rect x="{cx+dx-w/2:.1f}" y="{cy-h/2:.1f}" '
                f'width="{w:.1f}" height="{h:.1f}" '
                f'fill="{fill}" stroke="{stroke}" stroke-width="1" '
                f'rx="{w*0.2:.1f}" opacity="{opacity}"/>\n')
    return svg


@patch
def rot_arrow(self: DiagramGlyphs, cx, cy, r=None, clockwise=True,
              stroke=None, stroke_width=2, opacity=None):
    import math
    r = r or self.size * 0.5
    stroke = stroke or self.color
    opacity = opacity if opacity is not None else self.opacity
    if clockwise:
        arc_svg = self.arc(cx, cy, r, -120, 150, stroke, stroke_width, opacity)
        tip_x = cx + r * math.cos(math.radians(150))
        tip_y = cy + r * math.sin(math.radians(150))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x-r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x-r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    else:
        arc_svg = self.arc(cx, cy, r, 30, 300, stroke, stroke_width, opacity)
        tip_x = cx + r * math.cos(math.radians(30))
        tip_y = cy + r * math.sin(math.radians(30))
        ah = (f'<polygon points="'
              f'{tip_x:.1f},{tip_y:.1f} '
              f'{tip_x+r*0.25:.1f},{tip_y-r*0.15:.1f} '
              f'{tip_x+r*0.1:.1f},{tip_y+r*0.2:.1f}" '
              f'fill="{stroke}" opacity="{opacity}"/>\n')
    return arc_svg + ah
```

Key changes:

- **`DiagramGlyphs` dataclass** holds `color`, `stroke_width`, `opacity`, and `size` as defaults — every method falls back to `self.*` when a kwarg isn't passed.
- **`pause` now returns a string** instead of yielding — the old generator version was awkward to use in concatenation.
- **`rot_arrow` defaults `r`** from `self.size * 0.5` so you don't need to pass it for typical use.
- Every method follows the same `(self, cx, cy, ...)` signature so they compose naturally: `g = DiagramGlyphs(color="#4CAF50", size=12); g.shield(100, 200)`.

Then in `piece_plan_overlay` you'd create one instance and swap the old calls:

```python
g = DiagramGlyphs(color=color, size=grid.hexes[0].radius * 0.8)
# ...
svg += g.shield(cx, cy)           # was svg_shield(cx, cy, size, fill, stroke)
svg += g.rot_arrow(cx, cy, clockwise=True)  # was svg_rot_arrow(...)
```

def _action_glyph(instr: Instruction, cx, cy, size, fill, stroke) -> str:
    """Return SVG for a non-movement instruction. Pure geometry, no text."""
    if instr == Instruction.PAUSE:
        return "".join(svg_pause(cx, cy, size, fill, stroke))
    elif instr == Instruction.DEFEND:
        return svg_shield(cx, cy, size, fill, stroke)
    elif instr == Instruction.HARVEST:
        return svg_sheaf(cx, cy, size, fill, stroke)
    elif instr == Instruction.GIVE:
        return svg_gift(cx, cy, size, fill, stroke)
    elif instr == Instruction.SETTLE:
        return svg_house(cx, cy, size, fill, stroke)
    elif instr == Instruction.ROT_L:
        return svg_rot_arrow(cx, cy, size * 0.45, clockwise=False, stroke=stroke)
    elif instr == Instruction.ROT_R:
        return svg_rot_arrow(cx, cy, size * 0.45, clockwise=True, stroke=stroke)
    return ""


def piece_plan_overlay(steps: list[PieceStep], grid: HexGrid,
                       piece: Piece = None, c2f: dict = None,
                       piece_scale: float = None) -> str:
    """Render movement plan — pure SVG geometry, no emoji or text glyphs."""
    svg, N = "", len(grid.hexes)
    flag = piece.flag if piece else None
    r = grid.radius

    if piece_scale is None:
        piece_scale = r / 45.0

    color = flag.primary if flag else "#4CAF50"
    comp  = flag.comp    if flag else "#333"

    # Heraldic fill for chess pieces
    pat_fill = color
    if flag:
        pat_name = f"piece_pat_{piece.id[:8]}"
        pat = flag.flagPattern(pat_name, scale=0.1)
        grid.builder.add_definition(pat)
        pat_fill = f"url(#{pat_name})"

    glyph_size = max(6, r * 0.45)
    label_size = max(5, r * 0.25)
    dot_r      = max(2, r * 0.12)
    block_r    = max(3, r * 0.18)

    arrow_id = f"plan_arrow_{id(piece) % 9999}"
    arrow_style = StyleCSS(arrow_id,
                           stroke=color, stroke_width=max(1, r * 0.08),
                           fill="none", opacity="0.7",
                           stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}")
    grid.builder.add_style(arrow_style)

    # Start piece
    start_fine = _map_point(piece.location, c2f) if piece and piece.location is not None else -1
    if 0 <= start_fine < N and flag:
        center = grid.hexes[start_fine].center
        renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
        svg += getattr(flag, renderer)(MapCord(center.x, center.y),
                                       scale=piece_scale,
                                       piece_id=f"plan_start_{piece.id[:8]}",
                                       fill=pat_fill)
        # Start ring marker
        svg += svg_ring(center.x, center.y, r * 0.75, stroke=color, stroke_width=max(1.5, r*0.07))

    for step in steps:
        fine_idx  = _map_point(step.hex_idx, c2f)
        fine_prev = _map_point(step.prev_idx, c2f) if step.prev_idx >= 0 else -1
        if fine_idx < 0 or fine_idx >= N:
            continue
        cx, cy = grid.hexes[fine_idx].center.x, grid.hexes[fine_idx].center.y

        if (step.instruction == Instruction.FORWARD and not step.blocked
                and 0 <= fine_prev < N and fine_prev != fine_idx):
            svg += grid.arrow(fine_prev, fine_idx, style=arrow_style, factor=0.3)
            svg += svg_circle(cx, cy, dot_r, fill=color, stroke="none", opacity=0.55)

        elif step.blocked:
            # Red X using two crossed lines inside a circle
            svg += svg_circle(cx, cy, block_r, fill="red", stroke="darkred",
                              stroke_width=1, opacity=0.55)
            d = block_r * 0.55
            for x1,y1,x2,y2 in [(cx-d,cy-d,cx+d,cy+d),(cx+d,cy-d,cx-d,cy+d)]:
                svg += (f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" '
                        f'stroke="white" stroke-width="{max(1,block_r*0.35):.1f}" '
                        f'stroke-linecap="round"/>\n')

        else:
            svg += _action_glyph(step.instruction, cx, cy, glyph_size, color, comp)

        # Step number — only safe ASCII digits, no special chars
        off = r * 0.3
        svg += (f'<text x="{cx+off:.1f}" y="{cy-off:.1f}" text-anchor="start" '
                f'font-size="{label_size:.0f}" fill="{comp}" '
                f'font-family="sans-serif" font-weight="bold">{step.step+1}</text>\n')

    # Ghost piece at end
    if steps and flag:
        last = steps[-1]
        final_fine = _map_point(last.hex_idx, c2f)
        if 0 <= final_fine < N and final_fine != start_fine:
            end_center = grid.hexes[final_fine].center
            renderer = _PIECE_RENDERERS.get(piece.piece_type, 'pawnPiece')
            svg += getattr(flag, renderer)(MapCord(end_center.x, end_center.y),
                                           scale=piece_scale * 0.7,
                                           piece_id=f"plan_end_{piece.id[:8]}",
                                           fill=pat_fill,
                                           attrs={"opacity": "0.4"})

    return svg


def piece_plan_overlay(piece: Piece, grid: HexGrid,
                       steps: list[PieceStep] = None,
                       c2f: dict = None,
                       num_rounds: int = 12,
                       elevations: np.ndarray = None,
                       countries: np.ndarray = None) -> str:
    """Render movement plan overlay for a single piece.

    Pass pre-computed `steps`, or provide `elevations` to simulate on the fly.
    """
    if steps is None:
        if elevations is None:
            return ""
        steps = piece.simulate(grid, elevations, num_rounds, countries=countries)
    if not steps:
        return ""

    svg, N = "", len(grid.hexes)
    r = grid.radius
    color = piece.flag.primary if piece.flag else "#333"
    comp  = piece.flag.comp    if piece.flag else "#555"
    g = DiagramGlyphs(color=color, size=r * 0.45)

    # Dashed arrow style for movement lines
    arrow_style = StyleCSS(
        f"plan_arr_{piece.id[:8]}",
        stroke=color, stroke_width=max(1, r * 0.08),
        fill="none", opacity="0.7",
        stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}",
    )
    grid.builder.add_style(arrow_style)

    # ── Start: full piece + ring ──
    start = _map_point(piece.location, c2f)
    if 0 <= start < N:
        svg += piece.draw_svg(grid, hex_idx=start)
        c = grid.hexes[start].center
        svg += g.start_marker(c.x, c.y, r=r * 0.55)

    # ── Each step ──
    prev_idx = start
    for step in steps:
        idx = _map_point(step.hex_idx, c2f)
        if idx < 0 or idx >= N:
            prev_idx = idx
            continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        if step.instruction == Instruction.FORWARD and not step.blocked:
            if 0 <= prev_idx < N and prev_idx != idx:
                svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
            svg += piece.draw_ghost(grid, idx)

        elif step.blocked:
            svg += g.blocked_x(cx, cy)

        else:
            svg += g.action_glyph(step.instruction, cx, cy)

        svg += g.step_label(cx, cy, step.step, color=comp)
        prev_idx = idx

    return svg


Does piece_plan_overlay need to be revised?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — one naming bug and one simplification:

**1. `num_rounds` → `num_turns`** — `simulate` takes `num_turns` as its parameter name. You're passing it positionally so it *works*, but `_plan_overlay` on `PieceBoard` passes it as a keyword (`num_rounds=num_rounds`) which **will TypeError** (that's the exact error you hit earlier).

**2. `_plan_overlay` should just call `piece_plan_overlay`** — right now cell `_2a741198` reimplements the whole loop. It should be a thin wrapper.

Fix both:

```python
def piece_plan_overlay(piece: Piece, grid: HexGrid,
                       steps: list[PieceStep] = None,
                       c2f: dict = None,
                       num_turns: int = 12,          # ← renamed
                       elevations: np.ndarray = None,
                       countries: np.ndarray = None) -> str:
    """Render movement plan overlay for a single piece."""
    if steps is None:
        if elevations is None:
            return ""
        steps = piece.simulate(grid, elevations, num_turns=num_turns,
                               countries=countries)
    if not steps:
        return ""
    # ... rest unchanged ...
```

And `_plan_overlay` becomes:

```python
@patch
def _plan_overlay(self: PieceBoard, num_turns: int = 12) -> str:
    svg = ""
    for p in self.pieces:
        svg += piece_plan_overlay(
            p, self.grid, num_turns=num_turns,
            elevations=self.elevations, countries=self.countries,
        )
    return svg
```

Then update `render` to match:

```python
@patch
def render(self: PieceBoard, show_plan: bool = False,
           num_turns: int = 12, width: int = None, height: int = None) -> str:
    # ...
    grid.builder.adjust("plan", self._plan_overlay(num_turns) if show_plan else "")
    # ...
```

And `PieceBoardPlan`:

```python
def PieceBoardPlan(board: PieceBoard, num_turns: int = 12):
    return Div(NotStr(board.render(show_plan=True, num_turns=num_turns)))
```

That's it — just rename `num_rounds` → `num_turns` consistently everywhere and collapse the duplicated overlay loop.

#### swap down

Can you give me the full correct versions?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here are all the functions that need updating, fully corrected:

```python
def piece_plan_overlay(piece: Piece, grid: HexGrid,
                       steps: list[PieceStep] = None,
                       c2f: dict = None,
                       num_turns: int = 12,
                       elevations: np.ndarray = None,
                       countries: np.ndarray = None) -> str:
    """Render movement plan overlay for a single piece.

    Pass pre-computed `steps`, or provide `elevations` to simulate on the fly.
    """
    if steps is None:
        if elevations is None:
            return ""
        steps = piece.simulate(grid, elevations, num_turns=num_turns,
                               countries=countries)
    if not steps:
        return ""

    svg, N = "", len(grid.hexes)
    r = grid.radius
    color = piece.flag.primary if piece.flag else "#333"
    comp  = piece.flag.comp    if piece.flag else "#555"
    g = DiagramGlyphs(color=color, size=r * 0.45)

    # Dashed arrow style for movement lines
    arrow_style = StyleCSS(
        f"plan_arr_{piece.id[:8]}",
        stroke=color, stroke_width=max(1, r * 0.08),
        fill="none", opacity="0.7",
        stroke_dasharray=f"{r*0.25:.0f},{r*0.12:.0f}",
    )
    grid.builder.add_style(arrow_style)

    # ── Start: full piece + ring ──
    start = _map_point(piece.location, c2f)
    if 0 <= start < N:
        svg += piece.draw_svg(grid, hex_idx=start)
        c = grid.hexes[start].center
        svg += g.start_marker(c.x, c.y, r=r * 0.55)

    # ── Each step ──
    prev_idx = start
    for step in steps:
        idx = _map_point(step.hex_idx, c2f)
        if idx < 0 or idx >= N:
            prev_idx = idx
            continue
        cx, cy = grid.hexes[idx].center.x, grid.hexes[idx].center.y

        if step.instruction == Instruction.FORWARD and not step.blocked:
            if 0 <= prev_idx < N and prev_idx != idx:
                svg += grid.arrow(prev_idx, idx, style=arrow_style, factor=0.3)
            svg += piece.draw_ghost(grid, idx)

        elif step.blocked:
            svg += g.blocked_x(cx, cy)

        else:
            svg += g.action_glyph(step.instruction, cx, cy)

        svg += g.step_label(cx, cy, step.step, color=comp)
        prev_idx = idx

    return svg
```

```python
@patch
def _plan_overlay(self: PieceBoard, num_turns: int = 12) -> str:
    """SVG string: simulated movement plan for all pieces."""
    svg = ""
    for p in self.pieces:
        svg += piece_plan_overlay(
            p, self.grid, num_turns=num_turns,
            elevations=self.elevations, countries=self.countries,
        )
    return svg
```

```python
@patch
def render(self: PieceBoard, show_plan: bool = False,
           num_turns: int = 12, width: int = None, height: int = None) -> str:
    """Return a full SVG string of the board."""
    self._apply_terrain()
    grid = self.grid
    grid.update()

    grid.builder.adjust("pieces", self._pieces_overlay())
    grid.builder.adjust("plan", self._plan_overlay(num_turns) if show_plan else "")

    if width is None or height is None:
        xs = [h.center.x for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        ys = [h.center.y for i, h in enumerate(grid.hexes) if i not in grid.invalidRegion]
        pad = grid.radius * 1.5
        w = int(max(xs) + pad) if width  is None else width
        h = int(max(ys) + pad) if height is None else height
    else:
        w, h = width, height

    grid.builder.width  = w
    grid.builder.height = h
    return grid.builder.xml()
```

```python
@patch
def __ft__(self: PieceBoard):
    """Default view — terrain + pieces, no plan."""
    return Div(NotStr(self.render()))


def PieceBoardPlan(board: PieceBoard, num_turns: int = 12):
    """View with movement plan overlay."""
    return Div(NotStr(board.render(show_plan=True, num_turns=num_turns)))
```

```python
# ── Quick PieceBoard demo ─────────────────────────────────────────────────────
board = PieceBoard.hilly(rings=5, radius=22, seed=7)
grid  = board.grid
mid   = grid.middle

scout = board.add_piece(
    mid, piece_type=PieceType.KNIGHT, facing=3, country_id=1,
    rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_R.value,
    ],
)

worker = board.add_piece(
    mid - 3, piece_type=PieceType.PAWN, facing=4, country_id=1,
    rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.HARVEST.value, Instruction.HARVEST.value,
        Instruction.ROT_L.value, Instruction.ROT_L.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.GIVE.value,
    ],
)

guard = board.add_piece(
    mid + 7, piece_type=PieceType.ROOK, facing=0, country_id=2,
    rules=[
        Instruction.FORWARD.value,
        Instruction.ROT_R.value,
        Instruction.DEFEND.value, Instruction.DEFEND.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value,
    ],
)

bishop = board.add_piece(
    mid + grid.nCols * 2, piece_type=PieceType.BISHOP, facing=5, country_id=2,
    rules=[
        Instruction.FORWARD.value, Instruction.FORWARD.value,
        Instruction.ROT_L.value,
        Instruction.FORWARD.value,
        Instruction.PAUSE.value,
        Instruction.SETTLE.value,
    ],
)

show(board)
```

```python
PieceBoardPlan(board, num_turns=12)
```

The only substantive change across all of these is `num_rounds` → `num_turns` everywhere, plus `_plan_overlay` now delegates to `piece_plan_overlay` instead of reimplementing the loop.

### On Terrain

In [ ]:
showDemo = False

In [ ]:
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the existing piece (the knight)
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Assign a patrol route: forward-forward-turn, with a harvest stop
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
    Instruction.FORWARD.value,
    Instruction.ROT_L.value,
]
piece.patrol = True

# Simulate on the coarse grid
steps, region = piece.plan_region(
    grid, terrain.elevations,
    num_turns=6, padding=4,
    countries=terrain.fields.get("country")
)

print(f"Piece: {piece.piece_type.name} at hex {piece.location}, facing={piece.facing}")
print(f"{len(steps)} steps, {len(region)} hex region\n")
for s in steps:
    tag = " ✘" if s.blocked else ""
    print(f"  {s.step+1:2d}. {s.instruction.name:8s} → hex {s.hex_idx}  f={s.facing}{tag}")

# Zoom into the plan region
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw the plan overlay on the zoomed grid
overlay = piece_plan_overlay(piece, zoomed.hexGrid,
                             steps=steps,
                             c2f=result.c2f)

zoomed.hexGrid.builder.adjust("plan", overlay)
if showDemo:
    webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))


### On Text

In [ ]:
uid = ensure_user(dummySession)
active = globalStore.active_board(uid)
board, terrain, grid = active.board, active.board.terrain, active.board.terrain.hexGrid

# Find the knight
kingdom = next(k for k in board.kingdoms
               if any(c for s in k.settlements for c in s.citizens))
piece = next(c for s in kingdom.settlements for c in s.citizens)

# Set up rules: forward-forward-rotate-forward-forward-harvest (patrol + forage)
piece.rules = [
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.FORWARD.value, Instruction.FORWARD.value,
    Instruction.ROT_R.value,
    Instruction.HARVEST.value,
]
piece.patrol = True
piece.facing = 0

print(f"Piece: {piece.piece_type.name}, speed={piece.move_strength}, loc={piece.location}")
print(f"Rules: {[Instruction(r).name for r in piece.rules]}")
print(f"Patrol: {piece.patrol}\n")

# Simulate 5 turns
steps = piece.simulate(grid, terrain.elevations, num_turns=5,
                       countries=terrain.fields.get("country"))

# Print turn-by-turn
by_turn = steps_by_turn(steps)
for t in sorted(by_turn.keys()):
    print(f"── Turn {t+1} ──")
    for s in by_turn[t]:
        moved = f"→ hex {s.hex_idx}" if s.hex_idx != s.prev_idx else f"@ hex {s.hex_idx}"
        rot = f" (auto-rot {s.auto_rotations}×)" if s.auto_rotations else ""
        blk = " BLOCKED" if s.blocked else ""
        print(f"  step {s.step:2d}: {s.instruction.name:8s} {moved}  "
              f"face={s.facing}  cost={s.cost:.0f}  budget={s.budget_remaining:.0f}{rot}{blk}")
    print()

print(f"Total steps: {len(steps)}")
print(f"Final position: hex {steps[-1].hex_idx}, facing {steps[-1].facing}")


In [ ]:
# Re-simulate with the fix applied
steps, region = piece.plan_region(grid, terrain.elevations,
                                  num_turns=5, padding=4,
                                  countries=terrain.fields.get("country"))

print(f"{len(steps)} steps, {len(region)} hex region")

# Zoom
result = active.cover.zoom_region_fast(region, compute_weather=True)
zoomed = result.terrain
zoomed.hexGrid.adjustRadius(25)
zoomed.colorMap()
zoomed.hexGrid.update()

# Draw plan overlay
overlay = piece_plan_overlay(piece, zoomed.hexGrid,
                             steps=steps, c2f=result.c2f)

zoomed.hexGrid.builder.adjust("plan", overlay)
if showDemo:
    webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))


In [ ]:
#webMe(Div(NotStr(zoomed.hexGrid.builder.xml())))


In [ ]:
for s in steps:
    elev = terrain.elevations[s.hex_idx]
    ctry = terrain.fields["country"][s.hex_idx]
    tag = " ✘" if s.blocked else ""
    print(f"  {s.step+1:2d}. hex {s.hex_idx}  elev={elev:.1f}  country={ctry:.0f}{tag}")


## Put it together

#| export
# ── 1. Path → rules converter ─────────────────────────────────

def path_to_rules(hex_indices: list[int], grid: HexGrid, start_facing: int = 0) -> list[int]:
    """Convert a list of hex grid indices (from drag) into Instruction ints."""
    if len(hex_indices) < 2:
        return []

    dirs = HexPosition.directions()   # [SW, W, NW, NE, E, SE] = indices 0–5

    # Deduplicate consecutive duplicates (drag can repeat hexes)
    clean = [hex_indices[0]]
    for idx in hex_indices[1:]:
        if idx != clean[-1]:
            clean.append(idx)

    rules = []
    facing = start_facing

    for i in range(len(clean) - 1):
        from_pos = grid.index_to_hexposition(clean[i],   clean[0])
        to_pos   = grid.index_to_hexposition(clean[i+1], clean[0])
        delta    = to_pos - from_pos

        # Find which direction matches the delta
        try:
            target_dir = next(j for j, d in enumerate(dirs) if d == delta)
        except StopIteration:
            # Non-adjacent hex in drag — skip
            continue

        # Shortest rotation from facing to target_dir
        diff = (target_dir - facing) % 6
        if diff == 0:
            pass                                        # already facing right
        elif diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        rules.append(Instruction.FORWARD.value)
        facing = target_dir

    return rules

#| export
# ── 2. Rules → readable labels ────────────────────────────────

_INSTR_LABELS = {
    0: ("⏸ Pause",   "#94a3b8"),
    1: ("🛡 Defend",  "#ef4444"),
    2: ("🌾 Harvest", "#22c55e"),
    3: ("🎁 Give",    "#a855f7"),
    4: ("🏰 Settle",  "#f59e0b"),
    5: ("→ Forward", "#3b82f6"),
    6: ("↺ Rot L",   "#64748b"),
    7: ("↻ Rot R",   "#64748b"),
}

def rules_display(rules: list[int], piece_id: str):
    """Compact badge list for a rule sequence."""
    if not rules:
        return P("No rules yet — drag a path above", cls="text-sm opacity-50")

    badges = []
    for i, r in enumerate(rules):
        label, color = _INSTR_LABELS.get(r, (str(r), "#888"))
        badges.append(
            Span(label,
                 style=f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                       f"color:{color}; margin:2px; display:inline-block;")
        )

    return Div(
        P(f"{len(rules)} instructions", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        id=f"rules-{piece_id[:8]}",
    )

Should rules_display and path_to_rules be put on InstructionList? d

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Whoops! An error (prompt is too long: 231024 tokens > 200000 maximum) occurred while processing your request.
If this problem persists, please contact us on Discord.
Please include your dialog url and error info in your message.

#| export
### Revisit

In [ ]:
#| export
# ── 4. Route that receives the drag ───────────────────────────

@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return badge display."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # We need the grid to convert indices → HexPositions.
    # Use a fresh centered grid matching piece_drag_editor's rings/radius.
    rings  = 3
    radius = 35
    dummy_style = StyleCSS("x", fill="white")
    grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

    new_rules = path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist to DB if we can find the piece
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.get("rules") or "").split(",") if x]
            merged = existing + new_rules   # append; swap for replace if preferred
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in merged)},
                pk="id"
            )
            new_rules = merged
    except Exception:
        pass   # demo mode — just show without saving

    dummy = Piece(id=piece_id, rules=new_rules)
    return rules_display(dummy.rules, piece_id)

In [ ]:
#| export
dummy_rook = Piece(
    id   = "demo-rook-0000-0000-000000000000",
    piece_type = PieceType.ROOK,
    facing     = 4,   # starts facing E
    rules      = [],
)

webMe(Div(
    H3("Rook Rule Editor"),
    piece_drag_editor(dummy_rook),
    cls="p-4 space-y-4 bg-slate-900 text-white"
))

the drag-to-path converter. We could pratice with a dummy piece maybe a rook.

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret

In [ ]:
showActiveSettlements(dummySession)

# Simulate a drag path: center → E → NE → NE
grid = HexGrid.centered(3, radius=35, style=StyleCSS("x", fill="white"))
mid = grid.middle  # 24

# Manually pick a path: center, then one E, then one NE
e_idx  = grid.hexposition_to_index(HexPosition.E, mid)
ne_idx = grid.hexposition_to_index(HexPosition.NE, mid)
ne2    = grid.hexposition_to_index(HexPosition.NE + HexPosition.NE, mid)

drag_path = [mid, e_idx, ne_idx, ne2]
print(f"Drag indices: {drag_path}")

rules = path_to_rules(drag_path, grid, start_facing=4)  # facing E
labels = [_INSTR_LABELS[r][0] for r in rules]
print(f"Rules: {rules}")
print(f"Labels: {labels}")


Is it working?

Yes  wire up the /piece_editor_demo

#| export
def rules_display(rules: list[int], piece_id: str):
    """Compact badge list for a rule sequence, always with correct swap id."""
    target_id = f"rules-{piece_id[:8]}"

    if not rules:
        return Div(
            P("No rules yet — drag a path above", cls="text-sm opacity-50"),
            id=target_id,
        )

    badges = []
    for i, r in enumerate(rules):
        label, color = _INSTR_LABELS.get(r, (str(r), "#888"))
        badges.append(
            Span(label,
                 style=f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                       f"color:{color}; margin:2px; display:inline-block;")
        )

    # Count summary
    forwards = sum(1 for r in rules if r == 5)
    rotations = sum(1 for r in rules if r in (6, 7))
    actions = sum(1 for r in rules if r in (0, 1, 2, 3, 4))

    summary = f"{forwards} moves, {rotations} turns"
    if actions:
        summary += f", {actions} actions"

    return Div(
        P(f"{len(rules)} instructions — {summary}", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        Pre(",".join(str(r) for r in rules),
            cls="text-xs opacity-40 mt-2 font-mono"),
        id=target_id,
    )


### Revisit

In [ ]:
#| export
@rt("/piece_editor_demo")
def piece_editor_demo(session):
    """Live demo page for the drag-to-rules editor."""
    try:
        logging.info("piece_editor_demo: building page")
        editor = piece_drag_editor(dummy_rook)
        logging.info("piece_editor_demo: editor built OK")
        facing = facing_grid(dummy_rook.id, current_facing=dummy_rook.facing)
        logging.info("piece_editor_demo: facing built OK")
        return Titled("Piece Editor Demo",
            Div(
                H3("Rook Rule Editor"),
                P("Drag across hexes to create movement rules.",
                  cls="text-sm opacity-70 mb-4"),
                editor,
                Divider(),
                H4("Facing Selector"),
                facing,
                cls="p-6 space-y-6 max-w-xl mx-auto"
            )
        )
    except Exception as e:
        logging.error(f"piece_editor_demo FAILED: {e}", exc_info=True)
        return P(f"Error: {e}")


@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, return badge display."""
    try:
        form = await request.form()
        raw = form.get("hex_ids", "")
        logging.info(f"piece_path: piece_id={piece_id} facing={facing} raw='{raw}'")

        if not raw:
            return Div(P("No path received", cls="text-sm opacity-50"),
                       id=f"rules-{piece_id[:8]}")

        hex_ids = [int(x) for x in raw.split(",") if x.strip()]
        logging.info(f"piece_path: {len(hex_ids)} hex_ids: {hex_ids}")

        # Reconstruct same grid as piece_drag_editor
        rings  = 3
        radius = 35
        dummy_style = StyleCSS("x", fill="white")
        grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

        new_rules = path_to_rules(hex_ids, grid, start_facing=facing)
        logging.info(f"piece_path: {len(new_rules)} rules: {new_rules}")

        # For demo pieces, skip DB persistence
        if not piece_id.startswith("demo-"):
            try:
                rec = globalStore.pieces[piece_id]
                if rec:
                    existing_str = rec.rules if hasattr(rec, 'rules') else rec.get("rules", "")
                    existing = [int(x) for x in (existing_str or "").split(",") if x]
                    new_rules = existing + new_rules
                    globalStore.pieces.upsert(
                        {"id": piece_id, "rules": ",".join(str(r) for r in new_rules)},
                        pk="id"
                    )
            except Exception as db_err:
                logging.warning(f"piece_path: DB skip: {db_err}")

        return rules_display(new_rules, piece_id)

    except Exception as e:
        logging.error(f"piece_path FAILED: {e}", exc_info=True)
        return Div(P(f"Error: {e}", cls="text-sm text-red-500"),
                   id=f"rules-{piece_id[:8]}")


In [ ]:
!tail -10 base.text

This works and I am so excited. I want to do a commit which means I will loose the prompt history. can you write up where we have been and where we are going so I can put it into a note to come back to.

In [ ]:
homeSettle = [x for x in showActiveSettlements(dummySession) if x.id == "5642d99d-fbbe-4746-81ff-7e8f8552e3ca"][0]

In [ ]:
homeSettle

In [ ]:
knight =  homeSettle.citizens[0]

In [ ]:
knight

We need to adjust knight so that it has its parents_id  (from the '5642d99d-fbbe-4746-81ff-7e8f8552e3ca' and we should save that to the database as well

In [ ]:
knight.parent_id = homeSettle.id
knight.save(db=globalStore, world_id=1, kingdom_id=homeSettle.owner_id)
print(f"Knight parent_id set to {knight.parent_id[:12]}…")


In [ ]:
knight

I think some of the conversation about the simulator wasn't clear when I was doing the prompts. can you write up the approach I am aiming for so I can put it at the top. 